# Notebook 10 — fine-tune tahap 2 di dev resmi (latih convo 5, uji convo 2)

> [!info] Run berikutnya: `RUN_NOTE="tts-all"` (3 Okt): resep F, titik awal = stage 1 E4-native **+ TTS Central Javanese** (NB 14)
> NB 14 di dev: seri dengan E4-native (0,2339 vs 0,2317), tapi sisi Jawa jauh lebih baik (wer_jv 0,2425 → 0,2065)
> dan sisi Indonesia lebih buruk — kebalikan K (Homostoria, test 0,2698). Slot test yang memutuskan vs J 0,2655.
>
> Catatan run sebelumnya, `homo-all` (2 Okt) = arm K, test 0,2698: resep F, titik awal = stage 1 E4-native **+ Homostoria** (NB 13)
> NB 13 menurunkan dev greedy 0,2317 → 0,2096 (−0,0221 CI [−0,0268, −0,0177]). Dev tidak bisa menilai
> stage 1 (H), jadi slot test yang memutuskan melawan F 0,2660 / J 0,2655.
>
> Catatan run sebelumnya, `2ep-all` (1 Okt) = arm J: resep F, titik awal = stage 1 E4-native **2 epoch** (NB 12) → test 0,2655
> H (titik awal turbo) mendapat 0,3088 di test: stage 1 bernilai besar di test, jadi yang diuji sekarang
> stage 1 yang lebih panjang. Tanpa convo uji; slot test yang memutuskan melawan F 0,2660.
>
> Catatan run sebelumnya, `t0-all` (30 Sep) = arm H: resep F, titik awal turbo zero-shot
> `t0-c5` sudah jalan: convo 2 **0,1856 vs S-c5 0,1980**, −0,0124 CI [−0,0231, −0,0023] → stage 1
> merugikan. Run ini = varian `all` (convo 2 + 5, tanpa convo uji); slot test yang memutuskan vs F.
>
> Catatan run `t0-c5` (30 Sep): **titik awal turbo zero-shot, bukan D**
> NB 11 menemukan turbo zero-shot hanya +0,018 dari D di dev (ruang lower+tanpa tanda baca), dan di
> convo 2 malah **lebih baik** (0,2058 vs 0,2188). Pertanyaannya: sesudah tahap 2 di dev, apakah
> stage 1 (Jember + CV) masih menyumbang apa-apa? Resep = "c5" persis (convo 5 → uji convo 2,
> LR 3e-6); satu-satunya variabel adalah `BASE`. Kontrol "D" di sel 10 = titik awal = turbo
> zero-shot, jadi harus menyamai NB 11 (4.395 error). Pembanding sebenarnya, **S-c5 (0,1980 di
> convo 2)**, dihitung di laptop dari dua `submission_dev_S.csv`. Aturan keputusannya ada di sel 1.
> Riwayat run sebelumnya juga di sel 1 ("c5", "all" = F test 0,2660, "c5-lr6e6", "all-lr6e6" belum jalan).

**Pertanyaan.** Kalau D (E4-native) dilatih sebentar lagi pada data yang ditulis dalam
**konvensi referensi resmi**, apakah WER turun pada penutur yang tidak ikut dilatih?

**Kenapa.** Evaluasi gap #1 (`reports/gap-eval-1-reference-convention-2026-09-27.md`)
menemukan 66 % error varian ejaan memakai ejaan yang lebih sering di data latih kita
(Jember/CV): model meniru konvensi Jember. Satu-satunya data yang ditulis dalam konvensi
referensi resmi (keluaran omniASR_LLM_3B yang diedit manual) adalah dev resmi: 372 klip,
~2,6 jam, klip panjang code-switched seperti test.

> [!warning] Ini membalik keputusan 28 Agu: "dev hanya untuk validasi, tidak pernah untuk latih"
> Aturan kompetisi mengizinkannya (R9 hanya melarang anotasi / pseudo-label **test**), jadi
> itu keputusan strategi, bukan kepatuhan. Dibalik atas konfirmasi user 28 Sep, karena dev
> sudah terbukti proxy lemah untuk test. Validasi yang tersisa hanya convo 2 (held-out).

| | |
|---|---|
| Titik awal | **D** = E4-native `@3437f44` (test 0,2901 dengan beam 5) |
| Data latih | **convo 5**: 294 klip, 2 penutur, semuanya `javind`; klip > 30 s dibuang (~24) |
| Target | kolom `transcript` resmi apa adanya: case, tanda baca, `é`, `...` |
| Resep | 3 epoch, LR 3e-6 (setengah puncak tahap 1), warmup 10 %, linear, adafactor, batch efektif 16, SpecAugment. Sekitar 51 step |
| Checkpoint | step terakhir, tanpa seleksi |
| Evaluasi | **convo 2**: 78 klip, 2 penutur lain, 20 `ind` + 58 `javind`, 3.556 token. Lewat **`main.py` milik D, byte demi byte** (beam 5), **berpasangan** dengan D yang di-decode ulang di sesi yang sama. CI dari bootstrap per klip |

**Satu variabel terhadap D:** tahap 2 di convo 5. Satu perbedaan lingkungan dicatat: tahap 1
dilatih di transformers 5.0.0, notebook ini berjalan di `transformers<5` (versi container),
supaya decode-nya sama persis dengan `main.py`. D di-decode ulang di sini sebagai kontrol:
harus menyamai 3.965 error NB 08.

**Cara membaca hasil (ditetapkan sebelum run):**
- **JELAS LEBIH BAIK**: S − D di convo 2 negatif dan CI95-nya seluruhnya di bawah 0 →
  jalankan varian `RUN_NOTE="all"` (latih 372 klip, resep sama), lalu NB 08 dan slot test.
- **TIDAK JELAS** atau **LEBIH BURUK** → berhenti. Dev tidak dipakai untuk latih.

**Kenapa klip > 30 s dibuang, bukan dipotong.** Encoder Whisper melihat 30 s. Memotong
audio di 30 s tapi tetap memakai transkrip utuh mengajari model menulis kata yang tidak
terdengar, dan itu persis halusinasi yang ingin dihindari. Membelah butuh timestamp kata,
dan dev tidak punya. Yang dibuang ~24 klip, ~11 % audio convo 5.

**Batasan:** convo 2 cuma 2 penutur. CI bootstrap per klip meremehkan variasi antar
penutur, jadi "jelas lebih baik" di sini tetap bukan janji untuk test. Convo 2 juga berisi
**semua** 20 klip `ind`, sedangkan convo 5 tidak punya satu pun. Sel 11 melaporkan
subset `ind` dan `javind` terpisah.

## Setting Kaggle sebelum *Run All*

- **Accelerator:** GPU **T4** (satu dipakai).
- **Internet: ON sepanjang run.** Sel 1 memasang `transformers<5`, sel 3 mengunduh bobot,
  sel 9 push ke HF. Jangan ubah setting Internet di tengah sesi (sesi di-restart).
- **Secret `HF_TOKEN`** aktif di notebook ini (Add-ons → Secrets). Secret Kaggle berlaku
  per notebook. Tanpa token, sel 2 berhenti sebelum GPU dipakai.
- **Attach:** `indonesian-dev`.

**Waktu:** ± 35 menit di T4. Training ~10 menit, dua run `main.py` di 372 klip ~6 menit
masing-masing, sisanya unduh, push, dan kurva.

**Tanpa smoke terpisah.** Run penuhnya sendiri kira-kira seharga smoke. Titik gagal yang
murah diuji sebelum GPU dipakai lama: pin transformers + `TrainingArguments` (sel 1), HF
(sel 2), unduhan + sha (sel 3), sha `main.py` (sel 4), dan uji simpan checkpoint sebelum
training (sel 8). Bobot dipush **sebelum** evaluasi. Kalau sesi mati setelah push, set
`RUN_MODE="eval"` dan Run All: evaluasi diulang tanpa melatih lagi.

**Hasil:** `/kaggle/working/findings/10-dev-stage-finetune-c5/` plus blok
`CLAUDE-HANDOFF-JSON` di sel terakhir (agregat saja). Bobot ada di HF
(`amozaurus/mozilla-jv-10-dev-stage-finetune-c5`, private), tidak ikut di Output.

## 1 — Knob, lalu pin `transformers<5`

Semua yang menentukan run ini ada di satu blok. Pin terjadi sebelum transformers diimpor, sama seperti NB 08.

In [ ]:
# ============================================================
# CELL 1 - KNOBS, lalu pin transformers<5 SEBELUM impor apa pun
# ============================================================
import os, sys, subprocess
# Set sebelum torch diimpor. "T4 x2" memberi dua GPU, dan Trainer di dalam notebook lalu
# memakai DataParallel, yang gampang OOM di model 809 M (alasan lengkap: NB 06 sel 1).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# "train" = latih tahap 2, push ke HF, lalu evaluasi lewat main.py.
# "eval"  = tanpa training: unduh checkpoint S yang sudah dipush run ini, lalu evaluasi.
#           Dipakai kalau sesi mati SETELAH push (sel 9 mencetak revisi dan sha-nya).
RUN_MODE = "train"
EVAL_REVISION = None        # "eval": revisi repo HF run ini; None = main terbaru
EVAL_SHA16    = None        # "eval": sha256[:16] model.safetensors dari checkpoint_manifest.json

NOTEBOOK_ID = "10-dev-stage-finetune"
# Satu resep = satu RUN_NOTE = satu folder findings/ dan satu repo HF (CLAUDE.md aturan 3).
# Riwayat run notebook ini:
#   "c5"  (28 Sep): convo 5 -> uji convo 2, LR 3e-6. S - D = -0,0267 CI [-0,0390, -0,0142].
#   "all" (28 Sep): convo 2+5, LR 3e-6 = arm F di NB 08 -> TEST 0,2660 (submission 2914).
#   "c5-lr6e6" (run ini): SATU variabel vs "c5": LEARNING_RATE 3e-6 -> 6e-6 (= puncak tahap 1).
#       Pembanding utamanya "c5", bukan D: S(6e-6) - S(3e-6) di convo 2 dihitung di laptop dari
#       dua file submission_dev_S.csv (decode-nya deterministik; kontrol D membuktikan
#       lingkungannya sama). Aturan, ditetapkan sebelum run:
#         JELAS  : CI95 seluruhnya < 0 -> varian "all" dengan LR 6e-6, lalu NB 08 + slot test.
#         CONDONG: titik < 0 dan p(lebih baik) >= 0,90 -> diputuskan bersama, tergantung sisa slot.
#         selain itu: F (LR 3e-6) tetap yang terbaik.
# Varian "all" untuk LR apa pun: RUN_NOTE="all-<lr>", TRAIN_CONVOS=("2", "5"), EVAL_CONVOS=().
#   "c5-lr6e6" (29 Sep): lr6e6 - c5 di convo 2 = -0,0037 CI [-0,0101, +0,0028], p 0,857 -> dev
#       tidak bisa memutuskan. Karena masih ada 5 slot full, keputusannya dipindah ke test:
#   "all-lr6e6" (run ini) = arm G: resep "all" (F) dengan SATU variabel diubah, LR 6e-6.
#   "all-lr6e6" TIDAK dijalankan: P (29 Sep) menunjukkan tuas konvensi habis, dan rencana 30 Sep
#       memindahkan slot ke taruhan besar.
#   "t0-c5" (run ini, 30 Sep): resep "c5" persis, SATU variabel: BASE = turbo zero-shot, bukan D.
#       Pembanding = S-c5 (convo 2: 704 / 3556 = 0,1980), dihitung di laptop dari dua
#       submission_dev_S.csv. Aturan, ditetapkan sebelum run (S_T0 - S_c5 di convo 2):
#         CI95 seluruhnya < 0 -> stage 1 (Jember+CV) justru merugikan: varian "t0-all" + slot test.
#         CI95 seluruhnya > 0 -> stage 1 menyumbang: jalur E4-native tetap, lanjut R3 (2 epoch).
#         selain itu          -> stage 1 tidak terukur manfaatnya di convo 2: diputuskan bersama user.
#   "t0-c5" (30 Sep): convo 2 = 660 / 3556 = 0,1856 vs S-c5 704 -> S_T0 - S_c5 = -0,0124
#       CI95 [-0,0231, -0,0023] (lower+tanpa tanda baca -0,0132 [-0,0238, -0,0031]); kedua penutur
#       -0,012. Aturan: CI seluruhnya < 0 -> stage 1 (Jember+CV) merugikan -> varian "t0-all".
#   "t0-all" (run ini) = resep "all" (= F) dengan SATU variabel: BASE = turbo zero-shot.
#       Tanpa convo uji; yang memutuskan slot test melawan F 0,2660.
#   "t0-all" (30 Sep) = arm H -> TEST 0,3088 (submission 3159): convo 2 salah arah, stage 1 bernilai
#       >= 0,04 di test. Base turbo ditutup.
#   "2ep-all" (run ini, 1 Okt) = resep "all" (= F) persis, SATU variabel: BASE = stage 1 E4-native
#       2 epoch (NB 12 run 2). Tanpa convo uji; slot test yang memutuskan melawan F 0,2660.
#   "2ep-all" (1 Okt) = arm J -> TEST 0,2655 (submission 3376) vs F 0,2660: epoch ke-2 tidak membantu.
#   "homo-all" (run ini, 2 Okt) = resep "all" (= F) persis, SATU variabel: BASE = stage 1 E4-native
#       + Podcast Homostoria (NB 13). Dev greedy NB 13 0,2096 vs E4-native 0,2317, -0,0221 CI
#       [-0,0268, -0,0177]; tapi dev tidak bisa menilai stage 1 (H), jadi slot test yang memutuskan.
#   "homo-all" (2 Okt) = arm K -> TEST 0,2698 (submission 3473): dev stage 1 -0,0221 salah arah.
#   "tts-all" (run ini, 3 Okt) = resep "all" (= F) persis, SATU variabel: BASE = stage 1 E4-native
#       + TTS Central Javanese (NB 14). Dev greedy NB 14 0,2339 vs E4-native 0,2317 (+0,0022 CI
#       [-0,0025, +0,0066], seri); wer_jv 0,2425 -> 0,2065, wer_id 0,1754 -> 0,1934. Slot test memutuskan.
RUN_NOTE     = "tts-all"
TRAIN_CONVOS = ("2", "5")
EVAL_CONVOS  = ()

# Titik awal. Run "c5"/"all"/"c5-lr6e6" memakai D (E4-native, NB 06, 25 Sep):
#   BASE = dict(name="E4N", repo="amozaurus/mozilla-jv-06-data-mixing-E4-native",
#               rev="3437f4424cdee0ce3113f7fe2ccbc32e5350be7e", sha16="b3438f32cf61ebbe")
# Run "t0-c5": turbo zero-shot, revisi dan sha16 (fp16) persis seperti yang diukur NB 11.
#   Run "t0-c5"/"t0-all": BASE = dict(name="T0", repo="openai/whisper-large-v3-turbo",
#       rev="41f01f3fe87f28c78e2fbf8b568835947dd65ed9", sha16="542566a422ae4f3f")
# Run "2ep-all": stage 1 E4-native 2 epoch (NB 12 run 2, 1 Okt), step 842, fp32, tata letak transformers 5
# seperti D.
#   BASE = dict(name="E4N2", repo="amozaurus/mozilla-jv-12-stage1-2epoch-E4-native-2ep-r2",
#               rev="e4cd4ab4c8f707bb16d95c09cdfbb44c5c50e4f1", sha16="5cba4549f1b7f57e")
# Run "homo-all": stage 1 E4-native + Homostoria (NB 13, 2 Okt), step 714, fp32, tata letak transformers 5
# seperti D.
#   BASE = dict(name="E4NH", repo="amozaurus/mozilla-jv-13-stage1-homostoria-E4-native-homo",
#               rev="1bbab855c44e1d18aac52f86b50fd74ddc15cf07", sha16="dde745d5eff80bd8")
# Run "tts-all": stage 1 E4-native + TTS Central Javanese (NB 14, 3 Okt), step 506, fp32, tata letak
# transformers 5 seperti D.
BASE = dict(name="E4NT", repo="amozaurus/mozilla-jv-14-stage1-tts-E4-native-tts",
            rev="e892b99c36147ef13addc3ae1efe2ca239f2f63a", sha16="b494633b0708e5f9")

# Klip yang lebih panjang dari ini TIDAK dipakai melatih (alasan di markdown atas).
MAX_TRAIN_S = 30.0

# ---- resep tahap 2 ----
NUM_EPOCHS       = 3
LEARNING_RATE    = 3e-6       # = "c5" persis (c5-lr6e6 memakai 6e-6); variabel run ini hanya BASE
WARMUP_RATIO     = 0.10
PER_DEVICE_BATCH = 4
GRAD_ACCUM       = 4          # batch efektif 16, sama dengan tahap 1
OPTIM            = "adafactor"   # yang benar-benar dipakai tahap 1 (bitsandbytes tidak ada)
GRAD_CHECKPOINT  = True
SPEC_AUGMENT     = True       # regulariser tahap 1, nilai yang sama (sel 7)
SEED             = 20260829

# Kurva: greedy di dalam proses pada EVAL_CONVOS di epoch 0..N. OBSERVASI SAJA - yang
# dikirim tetap step terakhir. Gunanya memberi tahu varian "all" berapa epoch yang wajar.
CURVE     = True
NUM_BEAMS = 5                 # decode evaluasi lewat main.py = decode D
ALLOW_NO_HUB = False          # True = boleh latih tanpa HF (bobot hilang bersama sesi)

# ---- angka pembanding, dihitung di laptop dengan normalizer sel 5 (agregat saja) ----
# D = E4-native beam 5 lewat main.py (NB 08, findings/08-packaging-e4native/submission_dev_D.csv).
# Per convo: (error, token referensi).
# Run "t0-c5": kontrol "D" = titik awal = turbo zero-shot beam 5 lewat main.py, angka NB 11
# (findings/11-large-v3-probe-zeroshot-feasibility, arm T0). Angka D lama:
#   {"all": (3965, 17485), "2": (799, 3556), "5": (3166, 13929)}
D_REF = {"all": (4395, 17485), "2": (813, 3556), "5": (3582, 13929)}
N_REF = None                  # greedy titik awal belum pernah diukur (E4-native: 823 / 3556 di convo 2)
S_C5_REF = {"2": (704, 3556)} # S-c5 beam 5 di convo 2 = pembanding sebenarnya run ini
REF_PATHS = dict(short=328, chunked=44, windows=88)
BASE_MAIN_SHA16 = "8f76b71382f9b33c"   # main.py dengan NUM_BEAMS = 1 (NB 08)
D_MAIN_SHA16    = "aa5a691f68011b81"   # main.py D yang dikirim (submission 2693)
PARITY_TOL = 0.005
N_BOOT     = 5000
KNOWN = dict(D_test=0.2901, D_dev=0.2268, D_submission=2693, leaderboard_1=0.2096)

RUN_ID = NOTEBOOK_ID + "-" + RUN_NOTE
assert RUN_NOTE, "RUN_NOTE wajib diisi (CLAUDE.md aturan 3)"
assert RUN_MODE in ("train", "eval"), RUN_MODE
assert TRAIN_CONVOS, "TRAIN_CONVOS kosong"
assert not set(TRAIN_CONVOS) & set(EVAL_CONVOS), "convo latih dan convo uji harus terpisah"

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers<5"],
                   capture_output=True, text=True)
print("pip transformers<5 ->", "OK" if r.returncode == 0 else "GAGAL")
if r.returncode:
    print(r.stderr[-1500:])

import platform
import numpy, torch, transformers
try:
    import accelerate
    _acc = accelerate.__version__
except Exception as e:
    _acc = "tidak ada (%r)" % (e,)
print("python       :", platform.python_version())
print("transformers :", transformers.__version__, " (container: <5)")
print("accelerate   :", _acc)
print("torch        :", torch.__version__)
print("numpy        :", numpy.__version__)
print("gpu          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
MAJOR = int(transformers.__version__.split(".")[0])
if MAJOR >= 5:
    raise RuntimeError("transformers %s masih >= 5: pin gagal, atau transformers sudah diimpor "
                       "sebelum sel ini. Run -> Restart & clear, lalu Run All." % transformers.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("GPU mati. Settings -> Accelerator -> GPU T4, lalu Run All lagi.")

# Trainer di transformers 4.x belum pernah dipakai proyek ini (NB 04/06 melatih di 5.0.0).
# Membangun TrainingArguments sudah menyentuh accelerate; kalau versinya tidak cocok,
# gagalnya di sini, sebelum unduhan 3 GB.
from transformers import Seq2SeqTrainingArguments
_ta = Seq2SeqTrainingArguments(output_dir="/tmp/_targs_probe", report_to=[])
print("TrainingArguments: OK (%s)" % type(_ta).__name__)
del _ta
print("\nRUN_ID =", RUN_ID, "| mode", RUN_MODE, "| latih convo", TRAIN_CONVOS, "| uji convo", EVAL_CONVOS)

## 2 — Output, log, discovery, HuggingFace

Token dicek di sini: tanpa HF, bobot hasil latih hilang bersama sesi, jadi notebook berhenti sebelum GPU dipakai.

In [ ]:
# ============================================================
# CELL 2 - OUTPUT, LOG, DISCOVERY, HUGGINGFACE
# ============================================================
import json, re, csv, time, math, random, shutil, hashlib, collections, gc, inspect
from pathlib import Path
import numpy as np, pandas as pd
import warnings
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED)

def head(t):
    print("\n" + "=" * 74); print(t); print("=" * 74)

# Cermin /kaggle/working/findings/<RUN_ID>/ <-> <repo>/findings/<RUN_ID>/. Bobot TIDAK
# pernah masuk /kaggle/working (itu yang dipaketkan "Download all"); semuanya di /kaggle/temp.
WORKING  = Path("/kaggle/working")
TEMP     = Path("/kaggle/temp"); TEMP.mkdir(parents=True, exist_ok=True)
OUT_DIR  = WORKING / "findings" / RUN_ID
CKPT_DIR = TEMP / "ckpt" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def out(name):
    return OUT_DIR / name

# Semua yang dicetak juga masuk run-log.txt (pola NB 06).
_ORIG_STDOUT = getattr(sys, "_nb_orig_stdout", sys.stdout)
sys._nb_orig_stdout = _ORIG_STDOUT

class _Tee:
    def __init__(self, path, stream):
        self.f = open(path, "w", encoding="utf-8"); self.s = stream
    def write(self, t):
        self.s.write(t); self.f.write(t); return len(t)
    def flush(self):
        try: self.s.flush()
        except Exception: pass
        self.f.flush()
    def isatty(self): return False

LOG_PATH = out("run-log.txt")
try:
    if isinstance(sys.stdout, _Tee): sys.stdout.f.close()
except Exception:
    pass
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)

# Digest untuk Claude: AGREGAT SAJA. Tidak ada nama klip, transkrip, atau id penutur.
FINDINGS = {}
def rec(section, **kw):
    FINDINGS[section] = {**FINDINGS.get(section, {}), **kw}

def _num(x):
    x = float(x)
    if x == 0.0 or not math.isfinite(x):
        return x
    return float("%.6g" % x)

def jsonable(o):
    if isinstance(o, np.integer):  return int(o)
    if isinstance(o, np.floating): return _num(o)
    if isinstance(o, np.bool_):    return bool(o)
    if isinstance(o, float):       return _num(o)
    if isinstance(o, dict):        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v) for v in o]
    if isinstance(o, Path):        return str(o)
    return o

def dump_handoff():
    # Dicetak di sel 12, dan juga di sel yang gagal - Run All berhenti di sana.
    blob = json.dumps(jsonable(FINDINGS), ensure_ascii=False, separators=(",", ":"), sort_keys=True)
    print("\n===== BEGIN CLAUDE-HANDOFF-JSON =====")
    print(blob)
    print("===== END CLAUDE-HANDOFF-JSON =====")
    print("size: %.1f KB" % (len(blob) / 1024))

def sha16(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(1 << 22), b""): h.update(blk)
    return h.hexdigest()[:16]

head("DISCOVERY")
INPUT = Path("/kaggle/input")
_hits = sorted(INPUT.glob("**/indonesian_dev/metadata.tsv")) or \
        sorted(p for p in INPUT.glob("**/metadata.tsv") if (p.parent / "clips").is_dir())
if not _hits:
    raise FileNotFoundError("dev set tidak ditemukan - attach dataset indonesian-dev")
DEV_META  = _hits[0]
DEV_CLIPS = DEV_META.parent / "clips"
DEV = pd.read_csv(DEV_META, sep="\t", dtype=str, keep_default_na=False)
print("dev metadata :", DEV_META)
print("dev clips    :", len(list(DEV_CLIPS.glob("*.mp3"))), "mp3 | baris metadata:", len(DEV))
assert len(DEV) == 372, "dev resmi seharusnya 372 klip, dapat %d" % len(DEV)
assert set(DEV["audio_filename"]) == set(p.name for p in DEV_CLIPS.glob("*.mp3")), \
    "metadata dan clips/ tidak cocok satu-satu"
_cv = DEV["convo_id"].value_counts().sort_index()
print("convo_id     :", _cv.to_dict())
_missing = [c for c in tuple(TRAIN_CONVOS) + tuple(EVAL_CONVOS) if c not in set(DEV["convo_id"])]
assert not _missing, "convo_id tidak ada di metadata: %s" % _missing

head("HUGGINGFACE")
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        print("tidak ada secret HF_TOKEN (%r)" % (e,))

from huggingface_hub import HfApi
HF_ENABLED, HF_USER, HF_REPO, API = False, None, None, None
if HF_TOKEN:
    try:
        API = HfApi(token=HF_TOKEN)
        HF_USER = API.whoami()["name"]
        HF_REPO = "%s/mozilla-jv-%s" % (HF_USER, RUN_ID)
        if RUN_MODE == "train":
            API.create_repo(HF_REPO, private=True, exist_ok=True, repo_type="model")
        HF_ENABLED = True
        print("hf user : %s" % HF_USER)
        print("hf repo : %s  (private)" % HF_REPO)
    except Exception as e:
        print("!!! setup HF gagal: %r" % (e,))
if not HF_TOKEN:
    # Titik awalnya repo private: tanpa token bahkan unduhan di sel 3 gagal.
    raise RuntimeError("HF_TOKEN tidak ada. Add-ons -> Secrets -> HF_TOKEN (per notebook), lalu Run All.")
if RUN_MODE == "train" and not HF_ENABLED and not ALLOW_NO_HUB:
    raise RuntimeError("Hub mati, jadi bobot hasil latih akan hilang bersama sesi (kejadian 5 Sep). "
                       "Perbaiki HF_TOKEN, atau set ALLOW_NO_HUB=True kalau memang hanya mengukur.")

rec("env", notebook=NOTEBOOK_ID, run_id=RUN_ID, run_mode=RUN_MODE, run_note=RUN_NOTE,
    train_convos=list(TRAIN_CONVOS), eval_convos=list(EVAL_CONVOS),
    transformers=transformers.__version__, accelerate=_acc, torch=torch.__version__,
    numpy=np.__version__, python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
    hf_enabled=HF_ENABLED, hf_repo=HF_REPO, n_dev=len(DEV), convo_counts=_cv.to_dict())
rec("recipe", base=BASE, max_train_s=MAX_TRAIN_S, num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO, per_device_batch=PER_DEVICE_BATCH, grad_accum=GRAD_ACCUM,
    effective_batch=PER_DEVICE_BATCH * GRAD_ACCUM, optim=OPTIM, grad_checkpoint=GRAD_CHECKPOINT,
    spec_augment=SPEC_AUGMENT, seed=SEED, target="transcript (resmi, apa adanya)",
    num_beams_eval=NUM_BEAMS, curve=CURVE)
print("\noutput     :", OUT_DIR)
print("checkpoint :", CKPT_DIR, "(scratch, tidak ikut diunduh)")

## 3 — Checkpoint awal: D (E4-native)

Diunduh dalam fp32 (master weights untuk training) dan dicocokkan sha-nya dengan yang diukur NB 06 dan NB 08.

In [ ]:
# ============================================================
# CELL 3 - CHECKPOINT AWAL (D = E4-native) -> /kaggle/temp
# ============================================================
head("CHECKPOINT AWAL: %s" % BASE["name"])
from huggingface_hub import snapshot_download
from safetensors import safe_open

def _j(p):
    return json.loads(Path(p).read_text(encoding="utf-8")) if Path(p).exists() else {}

def fetch_checkpoint(repo, rev, dest):
    # Snapshot membawa pembukuan repo (.cache/, .gitattributes) dan training_args.bin,
    # yang tidak pernah dibaca saat inference. Sama dengan NB 08 sel 3.
    if dest.parent.exists():
        shutil.rmtree(dest.parent)
    dest.mkdir(parents=True)
    t0 = time.time()
    # Hanya file yang dibaca main.py / notebook ini. Repo OpenAI juga menyimpan bobot format
    # lain; tanpa filter itu ikut tersalin ke S sebagai "file non-bobot".
    snapshot_download(repo_id=repo, revision=rev, token=HF_TOKEN, local_dir=str(dest),
                      allow_patterns=["config.json", "generation_config.json", "model.safetensors",
                                      "tokenizer.json", "tokenizer_config.json", "preprocessor_config.json",
                                      "processor_config.json", "special_tokens_map.json", "added_tokens.json",
                                      "normalizer.json", "vocab.json", "merges.txt"])
    if (dest / ".cache").exists():
        shutil.rmtree(dest / ".cache")
    for p in list(dest.iterdir()):
        if p.is_file() and (p.name.startswith(".git") or p.name == "training_args.bin"):
            p.unlink()
    return time.time() - t0

BASE_DIR = TEMP / "ckpt_base" / "model"
_dl = fetch_checkpoint(BASE["repo"], BASE["rev"], BASE_DIR)
_got = sha16(BASE_DIR / "model.safetensors")
print("repo     : %s @ %s | unduh %.0f s" % (BASE["repo"], BASE["rev"][:12], _dl))
print("sha16    : %s %s" % (_got, "-> COCOK" if _got == BASE["sha16"] else "-> !!! TIDAK COCOK"))
assert _got == BASE["sha16"], "bobot yang diunduh bukan titik awal %s yang diukur" % BASE["name"]

BASE_FILES = {p.name: sha16(p) for p in sorted(BASE_DIR.iterdir()) if p.is_file()}
for n, h in BASE_FILES.items():
    print("   %-26s %9.1f MB  %s" % (n, (BASE_DIR / n).stat().st_size / 1e6, h))

# Hal-hal yang dibaca main.py sendiri dari checkpoint ini (disimpan transformers 5.0.0).
_proc = _j(BASE_DIR / "processor_config.json")
if (BASE_DIR / "preprocessor_config.json").exists():
    BASE_FE = _j(BASE_DIR / "preprocessor_config.json")
elif "feature_size" in _proc:
    BASE_FE = _proc
else:
    BASE_FE = next((v for v in _proc.values() if isinstance(v, dict) and "feature_size" in v), {})
BASE_TOKCFG = _j(BASE_DIR / "tokenizer_config.json")
CLEANUP = bool(BASE_TOKCFG.get("clean_up_tokenization_spaces", False))
with safe_open(str(BASE_DIR / "model.safetensors"), "pt") as f:
    BASE_KEYS = set(f.keys())
assert BASE_FE.get("feature_size") == 128, "feature extractor checkpoint bukan 128 mel"
print("\nfeature extractor : %s mel, sr %s | clean_up_tokenization_spaces: %s | %d tensor"
      % (BASE_FE.get("feature_size"), BASE_FE.get("sampling_rate"), CLEANUP, len(BASE_KEYS)))

rec("base", repo=BASE["repo"], revision=BASE["rev"], sha16=_got, sha_ok=(_got == BASE["sha16"]),
    files=BASE_FILES, n_tensors=len(BASE_KEYS), download_s=round(_dl, 1),
    saved_by=_j(BASE_DIR / "config.json").get("transformers_version"), cleanup=CLEANUP)

## 4 — `main.py` milik D

Salinan byte demi byte dari NB 08, dibuktikan lewat sha. Evaluasi di sel 10 memakai kode yang sama dengan yang menghasilkan 0,2901 di test.

In [ ]:
# ============================================================
# CELL 4 - main.py DAN postprocess.py, SALINAN BYTE DEMI BYTE DARI NB 08
# ============================================================
head("MAIN.PY")
# Evaluasi di sel 10 memakai main.py yang sama dengan yang dikirim untuk D. Satu-satunya
# baris yang diganti adalah NUM_BEAMS, dan sha-nya dicocokkan dengan main.py D (submission
# 2693). Jadi selisih S - D di sel 11 hanya bisa datang dari bobotnya.
SRC_MAIN = r'''# main.py - Lost in Transcription, Indonesian-Javanese track.
#
# Runs inside the organiser's container. Working directory is /code_execution:
#   data/clips/*.mp3        read-only test audio
#   data/test_metadata.csv  audio_filename, file_duration_seconds, language
#   src/                    this file, postprocess.py and ./model
#   submission/             where submission.csv must be written
#
# Constraints this file is written against, all taken from the competition's
# own pages rather than assumed:
#   - no network of any kind, so weights ship in ./model and offline flags are
#     set BEFORE transformers is imported (it reads them at import time)
#   - 2 h for the full set, 1 min for the 5-clip smoke test: the smoke budget is
#     import + weight load, not decode, so nothing heavy happens at module scope
#   - logs are capped at 500 lines x 300 chars AND logging anything about the
#     test data is grounds for disqualification, so only aggregate counts and
#     timings are printed - never a filename, a duration, or a transcript
#   - each clip is processed independently (competition rule 4): no state, no
#     sorting by content, no cross-clip normalisation
#
# The decode path is notebook 06's evaluation code (its cells 1 and 7), constant
# for constant: the dev WER this model was chosen on only carries over to the
# test set if the submission decodes exactly the same way.
import os

# Must precede the transformers/HF imports below.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Library warnings go to stderr, which counts against the same 500-line cap.
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

import collections
import csv
import json
import math
import sys
import time
import traceback
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

_T_IMPORT_START = time.time()
import numpy as np
import torch
from transformers import (GenerationConfig, WhisperFeatureExtractor,
                          WhisperForConditionalGeneration, WhisperTokenizerFast)

try:
    import librosa
except Exception:
    librosa = None
_T_IMPORT = time.time() - _T_IMPORT_START

HERE = Path(__file__).resolve().parent
WORK = Path.cwd()
DATA = WORK / "data"
CLIPS = DATA / "clips"
OUT_DIR = WORK / "submission"
MODEL_DIR = HERE / "model"

# Notebook 06, cell 1 (knobs) and cell 7 (decode helpers).
SR = 16000
LANG = "jw"                 # Whisper's Javanese code is "jw", not ISO "jv"
GEN_MAX_LENGTH = 225
NO_REPEAT_NGRAM = 4
BATCH = 8                   # EVAL_BATCH
LONG_S = 29.8               # clips longer than this take the chunked path
CHUNK_S = 28.0              # LONG_CHUNK_S
OVERLAP_S = 2.0             # LONG_OVERLAP_S
MIN_WINDOW_S = 0.3          # a trailing window this short is dropped
MAX_OVERLAP_WORDS = 8       # seam de-duplication, longest match first

# Beam width. 1 = greedy, the setting notebook 06 measured and every constant
# above was chosen under. The packaging notebook rewrites this one line per
# experiment arm and refuses an arm whose main.py differs anywhere else.
NUM_BEAMS = 1

GEN_KWARGS = dict(task="transcribe", language=LANG, num_beams=NUM_BEAMS, do_sample=False,
                  no_repeat_ngram_size=NO_REPEAT_NGRAM, max_length=GEN_MAX_LENGTH,
                  return_timestamps=False)

# A few unreadable clips cost a few points. A run where many clips fail means
# something systemic broke - a library mismatch, a missing file - and would
# hand the platform a scored file of blanks that burns a submission. Past this
# rate no submission.csv is written and the run exits non-zero; failed runs do
# not count against the submission limit.
MAX_FAIL_RATE = 0.02
MAX_FAIL_FLOOR = 2
FAIL_FAST_AFTER = 16

# Set only by our own test notebook. Never set on the platform, where a
# traceback could say something about a test file.
DEBUG = os.environ.get("LIT_DEBUG") == "1"


def log(msg):
    # Aggregate facts only. Never clip identities or transcript content.
    print(msg, flush=True)


def load_audio(path):
    # Notebook 06's load_official_audio: librosa first, soundfile + polyphase
    # resampling as the fallback. The same loader and the same resampler mean
    # the model sees the samples it was measured on.
    if librosa is not None:
        try:
            y, _ = librosa.load(str(path), sr=SR, mono=True)
            return y.astype(np.float32)
        except Exception:
            pass
    import soundfile as sf
    y, sr0 = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr0 != SR:
        from scipy.signal import resample_poly
        g = math.gcd(int(sr0), SR)
        y = resample_poly(y, SR // g, int(sr0) // g)
    return y.astype(np.float32)


def split_windows(y):
    # Fixed windows with an overlap, so a word straddling a boundary is seen
    # whole by at least one window. Whisper's encoder window is 30 s and
    # anything past it would be silently cut.
    step = int((CHUNK_S - OVERLAP_S) * SR)
    win = int(CHUNK_S * SR)
    parts = [y[i:i + win] for i in range(0, len(y), step)]
    return [p for p in parts if len(p) > int(MIN_WINDOW_S * SR)]


def join_overlapping(texts):
    # Consecutive windows usually repeat a few words at the seam. Drop the
    # longest repeated run, longest match first.
    joined = ""
    for t in texts:
        t = (t or "").strip()
        if not t:
            continue
        if not joined:
            joined = t
            continue
        a, b = joined.split(), t.split()
        k = 0
        for n in range(min(MAX_OVERLAP_WORDS, len(a), len(b)), 0, -1):
            if [w.lower() for w in a[-n:]] == [w.lower() for w in b[:n]]:
                k = n
                break
        joined = " ".join(a + b[k:])
    return joined.strip()


def _read_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


FE_KEYS = ("feature_size", "sampling_rate", "hop_length", "chunk_length", "n_fft",
           "padding_value", "dither")


def feature_extractor_config():
    # The checkpoint was saved by transformers 5, which writes a single
    # processor_config.json with the feature extractor nested inside it.
    # transformers 4 - the version the runtime pins - expects a separate
    # preprocessor_config.json. Reading the values by hand makes the file
    # layout of either major version irrelevant.
    pre = MODEL_DIR / "preprocessor_config.json"
    cfg = _read_json(pre) if pre.exists() else _read_json(MODEL_DIR / "processor_config.json")
    if "feature_size" in cfg:
        return cfg
    for v in cfg.values():
        if isinstance(v, dict) and "feature_size" in v:
            return v
    raise RuntimeError("no feature extractor settings found in ./model")


class Recogniser:
    def __init__(self):
        t0 = time.time()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32

        fe_cfg = feature_extractor_config()
        self.fe = WhisperFeatureExtractor(**{k: fe_cfg[k] for k in FE_KEYS if k in fe_cfg})
        # Built straight from tokenizer.json, which both major versions write in
        # the same format; tokenizer_config.json is the file whose format moved.
        # The only thing decoding needs from it is the clean-up flag.
        self.tok = WhisperTokenizerFast(tokenizer_file=str(MODEL_DIR / "tokenizer.json"))
        tok_cfg_path = MODEL_DIR / "tokenizer_config.json"
        tok_cfg = _read_json(tok_cfg_path) if tok_cfg_path.exists() else {}
        self.cleanup = bool(tok_cfg.get("clean_up_tokenization_spaces", False))

        model, info = WhisperForConditionalGeneration.from_pretrained(
            str(MODEL_DIR), local_files_only=True, dtype=self.dtype, output_loading_info=True)
        # With library warnings silenced, a checkpoint whose tensor names do not
        # match would load as random weights without a word. Refuse instead. Two
        # tensors may legitimately be absent from a file saved by another major
        # version: the output projection (tied to the token embedding) and the
        # encoder's fixed sinusoidal positions, which the library recomputes.
        derived = ("proj_out.weight", "encoder.embed_positions.weight")
        missing = info.get("missing_keys", [])
        self.n_derived = sum(1 for k in missing if k.endswith(derived))
        bad = len(missing) - self.n_derived + len(info.get("mismatched_keys", []))
        if bad:
            raise RuntimeError("checkpoint does not fit the model: %d tensors missing or mismatched" % bad)
        self.n_unexpected = len(info.get("unexpected_keys", []))
        # The explicit cast makes the dtype independent of which keyword this
        # transformers release honours (dtype vs torch_dtype).
        self.model = model.to(device=self.device, dtype=self.dtype).eval()
        self.model.config.use_cache = True
        if getattr(self.model.config, "forced_decoder_ids", None) is not None:
            self.model.config.forced_decoder_ids = None
        if self.fe.feature_size != self.model.config.num_mel_bins:
            raise RuntimeError("feature extractor has %d mel bins, model expects %d"
                               % (self.fe.feature_size, self.model.config.num_mel_bins))

        # generation_config.json carries Whisper's language table; without it
        # language="jw" cannot be honoured. Loaded explicitly so that a file this
        # release cannot parse stops the run instead of silently decoding with a
        # default config.
        gc = GenerationConfig.from_pretrained(str(MODEL_DIR), local_files_only=True)
        if "<|%s|>" % LANG not in (getattr(gc, "lang_to_id", None) or {}):
            raise RuntimeError("generation_config.json has no <|%s|> entry" % LANG)
        # Notebook 06's apply_gen_config: language and task pinned, the legacy
        # forced_decoder_ids path switched off so it cannot silently win.
        gc.language = LANG
        gc.task = "transcribe"
        gc.forced_decoder_ids = None
        gc.num_beams = NUM_BEAMS
        gc.do_sample = False
        gc.no_repeat_ngram_size = NO_REPEAT_NGRAM
        gc.max_length = GEN_MAX_LENGTH
        gc.use_cache = True
        self.model.generation_config = gc
        self.t_load = time.time() - t0

    @torch.no_grad()
    def transcribe(self, arrays):
        feats = self.fe(arrays, sampling_rate=SR, return_tensors="pt").input_features
        ids = self.model.generate(feats.to(self.device, dtype=self.dtype), **GEN_KWARGS)
        return [t.strip() for t in self.tok.batch_decode(
            ids, skip_special_tokens=True, clean_up_tokenization_spaces=self.cleanup)]


def read_names():
    meta_path = DATA / "test_metadata.csv"
    if meta_path.exists():
        with open(meta_path, newline="", encoding="utf-8") as fh:
            return [r["audio_filename"] for r in csv.DictReader(fh)]
    # The manifest is documented, but walking clips/ costs nothing and means
    # a missing or renamed manifest degrades instead of failing the run.
    return sorted(p.name for p in CLIPS.glob("*.mp3"))


def main():
    t_start = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    names = read_names()
    log("clips to transcribe: %d" % len(names))

    rec = Recogniser()
    log("import %.1fs | model load %.1fs | device %s | %s | tensors recomputed %d, unexpected %d"
        % (_T_IMPORT, rec.t_load, rec.device, str(rec.dtype).replace("torch.", ""),
           rec.n_derived, rec.n_unexpected))

    from postprocess import postprocess

    results = {}
    stats = collections.Counter()
    errors = collections.Counter()
    buf_names, buf_audio = [], []

    def failed(exc):
        stats["failed"] += 1
        errors[type(exc).__name__] += 1
        if DEBUG and errors[type(exc).__name__] == 1:
            traceback.print_exc()

    def run_short():
        # One generate() call per BATCH short clips in manifest order - the batch
        # composition notebook 06 measured. If the batch throws, each clip is
        # retried alone so one bad input cannot blank its neighbours.
        try:
            texts = rec.transcribe(buf_audio)
        except Exception:
            texts = []
            for a in buf_audio:
                try:
                    texts.append(rec.transcribe([a])[0])
                except Exception as e:
                    failed(e)
                    texts.append(None)
        for nm, tx in zip(buf_names, texts):
            results[nm] = "" if tx is None else postprocess(tx)
        stats["short"] += len(buf_names)
        buf_names.clear()
        buf_audio.clear()

    for name in names:
        try:
            y = load_audio(CLIPS / name)
        except Exception as e:
            failed(e)
            results[name] = ""
        else:
            if len(y) / SR > LONG_S:
                # Decoded on the spot, without flushing the short-clip buffer
                # first: notebook 06 did the same, so batches line up clip for clip.
                try:
                    parts = split_windows(y)
                    results[name] = (postprocess(join_overlapping(rec.transcribe(parts)))
                                     if parts else "")
                    stats["chunked"] += 1
                    stats["windows"] += len(parts)
                except Exception as e:
                    failed(e)
                    results[name] = ""
            else:
                buf_names.append(name)
                buf_audio.append(y)
                if len(buf_names) == BATCH:
                    run_short()
        if len(results) >= FAIL_FAST_AFTER and stats["failed"] == len(results):
            raise RuntimeError("the first %d clips all failed (%s) - stopping before a blank file"
                               " is scored" % (len(results), ", ".join(sorted(errors))))
    if buf_names:
        run_short()

    n = len(names)
    n_empty = sum(1 for nm in names if not results.get(nm, "").strip())
    log("short %d | chunked %d (%d windows) | failed %d | empty %d | %d clips in %.1fs"
        % (stats["short"], stats["chunked"], stats["windows"], stats["failed"], n_empty,
           n, time.time() - t_start))
    if errors:
        log("failure types: %s" % ", ".join("%s x%d" % kv for kv in sorted(errors.items())))
    if stats["failed"] > max(MAX_FAIL_FLOOR, int(MAX_FAIL_RATE * n)):
        log("too many failed clips - no submission.csv written, exiting non-zero")
        return 1

    # csv.writer, not string formatting: transcripts contain commas and quotes,
    # and the platform rejects a hand-assembled file.
    out = OUT_DIR / "submission.csv"
    with open(out, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["audio_filename", "transcript"])
        for nm in names:
            w.writerow([nm, results.get(nm, "")])
    log("wrote %d rows" % n)
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''
SRC_POST = r'''# postprocess.py - text fixes applied to every hypothesis before it is written.
#
# DELIBERATELY ALMOST EMPTY IN PHASE 1. The orthographic rules belong to
# notebook 07, and each one has to earn its place with a paired bootstrap on the
# official dev set before it goes in here. A rule that "looks obviously right"
# is how a submission quietly gets worse: the E4 error table shows the reference
# disagreeing with itself in both directions (gak->nggak 62 times, nggak->gak 33
# times), so a blanket rewrite fixes one column and breaks the other.
#
# What IS safe now is whitespace: it cannot change a token, only how tokens are
# separated, and the official scorer collapses runs of spaces anyway.
#
# Notebook 07 fills RULES below. Keep every entry measurable in isolation.
RULES = []


def postprocess(text):
    if not text:
        return ""
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    for pattern, replacement in RULES:
        text = pattern.sub(replacement, text)
    return " ".join(text.split()).strip()
'''
import ast
ast.parse(SRC_MAIN); ast.parse(SRC_POST)

def _sha_text(t):
    return hashlib.sha256(t.encode()).hexdigest()[:16]

MAIN_EVAL, _n_sub = re.subn(r"^NUM_BEAMS = 1$", "NUM_BEAMS = %d" % NUM_BEAMS, SRC_MAIN, flags=re.M)
assert _n_sub == 1, "baris NUM_BEAMS di main.py tidak ditemukan tepat satu kali"
MAIN_BASE_SHA, MAIN_EVAL_SHA = _sha_text(SRC_MAIN), _sha_text(MAIN_EVAL)
print("main.py dasar  sha16 %s  (NB 08: %s) -> %s"
      % (MAIN_BASE_SHA, BASE_MAIN_SHA16, "SAMA" if MAIN_BASE_SHA == BASE_MAIN_SHA16 else "!!! BEDA"))
print("main.py beam %d sha16 %s  (D dikirim: %s) -> %s"
      % (NUM_BEAMS, MAIN_EVAL_SHA, D_MAIN_SHA16,
         "SAMA" if MAIN_EVAL_SHA == D_MAIN_SHA16 else "beda (NUM_BEAMS != 5?)"))
assert MAIN_BASE_SHA == BASE_MAIN_SHA16, "salinan main.py tidak identik dengan NB 08"
if NUM_BEAMS == 5:
    assert MAIN_EVAL_SHA == D_MAIN_SHA16, "main.py evaluasi tidak identik dengan main.py D"
rec("source", main_base_sha16=MAIN_BASE_SHA, main_eval_sha16=MAIN_EVAL_SHA,
    same_as_shipped_D=(MAIN_EVAL_SHA == D_MAIN_SHA16), num_beams=NUM_BEAMS)

## 5 — Scorer resmi dan bootstrap berpasangan

`normalize_text` dari `score.py` penyelenggara, apa adanya, plus ruang kedua (huruf kecil, tanpa tanda baca) untuk memisahkan efek kosmetik.

In [ ]:
# ============================================================
# CELL 5 - SCORER RESMI + BOOTSTRAP BERPASANGAN (salinan NB 08 sel 8)
# ============================================================
# Disalin apa adanya dari score.py di runtime repo penyelenggara. JANGAN diedit
# agar cocok dengan scorer.py kita - arah pembuktiannya justru sebaliknya.
def _lowercase_sentence_initial(m):
    delimiter, first, second = m.group(1), m.group(2), m.group(3)
    if first.isupper() and not (second and second.isupper()): first = first.lower()
    return delimiter + first + second

def normalize_text(text):
    BRACKETED = re.compile(r"\[[^\]]+\]"); UNINTELLIGIBLE_PAREN = re.compile(r"\(\?+\)")
    WORD_PAREN = re.compile(r"\(([^()]*)\)"); PUNCTUATION_OTHER = re.compile('[¿¡";:]+')
    COMMA = re.compile(",+"); SENTENCE_INITIAL = re.compile(r"(^\s*|[.!?—]\s*)([^\W\d_])([^\W\d_]?)")
    SENTENCE_END = re.compile("[!?]+"); MULTISPACE = re.compile("  +")
    text = text.replace("~", "")
    text = re.sub(BRACKETED, " ", text); text = re.sub(UNINTELLIGIBLE_PAREN, " ", text)
    text = re.sub(WORD_PAREN, r"\1", text); text = text.replace("#x27;", "'")
    text = re.sub(PUNCTUATION_OTHER, " ", text)
    text = re.sub(SENTENCE_INITIAL, _lowercase_sentence_initial, text)
    text = text.replace("—", ", "); text = re.sub(COMMA, " ", text)
    text = re.sub(SENTENCE_END, " ", text)
    text = text.replace("...", "!ELLIPSIS!").replace(".", " ").replace("!ELLIPSIS!", "...")
    while " ... " in text: text = text.replace(" ... ", " ")
    return re.sub(MULTISPACE, " ", text)

def _lev(r, h):
    prev = list(range(len(h) + 1))
    for i, a in enumerate(r, 1):
        cur = [i] + [0] * len(h)
        for j, b in enumerate(h, 1):
            cur[j] = prev[j - 1] if a == b else 1 + min(prev[j - 1], prev[j], cur[j - 1])
        prev = cur
    return prev[-1]

def corpus_wer(refs, hyps):
    e = n = 0
    for r, h in zip(refs, hyps):
        rt, ht = normalize_text(r).split(), normalize_text(h).split()
        e += _lev(rt, ht); n += len(rt)
    return e / max(n, 1), e, n

def clip_errors(refs, hyps):
    # Per klip: (error, token referensi). Jumlahnya sama dengan corpus_wer; versi per
    # klip dibutuhkan untuk bootstrap berpasangan di bawah.
    out = []
    for r, h in zip(refs, hyps):
        rt, ht = normalize_text(r).split(), normalize_text(h).split()
        out.append((_lev(rt, ht), len(rt)))
    return np.array(out, dtype=np.int64).reshape(-1, 2)

def paired(ex, ey, n_boot=N_BOOT, seed=20260924):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(ex), size=(n_boot, len(ex)))
    tok = ex[:, 1][idx].sum(1)
    dd = (ex[:, 0][idx].sum(1) - ey[:, 0][idx].sum(1)) / np.maximum(tok, 1)
    lo, hi = np.percentile(dd, [2.5, 97.5])
    point = (ex[:, 0].sum() - ey[:, 0].sum()) / max(ex[:, 1].sum(), 1)
    return dict(delta=round(float(point), 4), ci95=[round(float(lo), 4), round(float(hi), 4)],
                p_better=round(float((dd < 0).mean()), 3))

def lower_nopunct(t):
    # Ruang pembanding kedua: normalizer resmi, lalu huruf kecil semua dan tanpa tanda
    # baca. Kalau S unggul di ruang resmi tapi tidak di sini, keunggulannya kosmetik
    # (case / tanda baca), bukan kata yang didengar.
    return " ".join(re.sub(r"[^\w\s'-]", " ", normalize_text(t).lower()).split())

def clip_errors_ln(refs, hyps):
    out_ = []
    for r, h in zip(refs, hyps):
        rt, ht = lower_nopunct(r).split(), lower_nopunct(h).split()
        out_.append((_lev(rt, ht), len(rt)))
    return np.array(out_, dtype=np.int64).reshape(-1, 2)

REF = dict(zip(DEV["audio_filename"], DEV["transcript"]))
print("scorer resmi siap | contoh normalisasi:", repr(normalize_text("Terus... Aku, ya? Ndak!")))

## 6 — Data: audio dev ke RAM, split per `convo_id`

Loader yang sama dengan `main.py`. Klip > 30 s di convo latih dibuang; convo uji tidak disentuh.

In [ ]:
# ============================================================
# CELL 6 - AUDIO DEV KE RAM, SPLIT LATIH / UJI PER convo_id
# ============================================================
head("DATA")
import librosa
SR = 16000

# Loader yang sama dengan main.py (librosa, resample ke 16 kHz): model dilatih pada sampel
# yang sama persis dengan yang nanti ia dengar saat decode. ~2,6 jam float32 = ~0,6 GB RAM.
AUDIO, DUR = {}, {}
_t0 = time.time()
for _i, _f in enumerate(DEV["audio_filename"]):
    _y, _ = librosa.load(str(DEV_CLIPS / _f), sr=SR, mono=True)
    AUDIO[_f] = _y.astype(np.float32)
    DUR[_f] = len(_y) / SR
    if (_i + 1) % 100 == 0:
        print("  %d/%d klip, %.0f s" % (_i + 1, len(DEV), time.time() - _t0))
print("audio: %d klip, %.2f jam, %.0f s" % (len(AUDIO), sum(DUR.values()) / 3600, time.time() - _t0))
assert all(DUR[f] > 0 for f in DUR), "ada klip yang ter-decode kosong"

DEV["dur_s"] = DEV["audio_filename"].map(DUR)
IS_TRAIN = DEV["convo_id"].isin(TRAIN_CONVOS)
IS_EVAL  = DEV["convo_id"].isin(EVAL_CONVOS)
TOO_LONG = DEV["dur_s"] > MAX_TRAIN_S
TRAIN_DF = DEV[IS_TRAIN & ~TOO_LONG].reset_index(drop=True)
DROP_DF  = DEV[IS_TRAIN & TOO_LONG].reset_index(drop=True)
EVAL_DF  = DEV[IS_EVAL].reset_index(drop=True)     # urutan metadata
EVAL_NAMES = list(EVAL_DF["audio_filename"])

print("\n%-8s %5s %7s %7s %7s %6s %6s  %s" % ("convo", "klip", "jam", "mean s", "max s", ">29.8", ">30", "peran"))
_convo_stats = {}
for c, g in DEV.groupby("convo_id"):
    role = "LATIH" if c in TRAIN_CONVOS else ("UJI" if c in EVAL_CONVOS else "-")
    st = dict(n=len(g), hours=round(g["dur_s"].sum() / 3600, 3), mean_s=round(g["dur_s"].mean(), 2),
              max_s=round(g["dur_s"].max(), 2), n_over_29_8=int((g["dur_s"] > 29.8).sum()),
              n_over_30=int((g["dur_s"] > 30.0).sum()), n_speakers=int(g["speaker"].nunique()),
              languages=g["language"].value_counts().to_dict(), role=role)
    _convo_stats[c] = st
    print("%-8s %5d %7.2f %7.1f %7.1f %6d %6d  %s, %d penutur, %s"
          % (c, st["n"], st["hours"], st["mean_s"], st["max_s"], st["n_over_29_8"], st["n_over_30"],
             role, st["n_speakers"], st["languages"]))

print("\ndata latih : %d klip, %.2f jam (dibuang karena > %.0f s: %d klip, %.2f jam = %.1f%% audio convo latih)"
      % (len(TRAIN_DF), TRAIN_DF["dur_s"].sum() / 3600, MAX_TRAIN_S, len(DROP_DF),
         DROP_DF["dur_s"].sum() / 3600,
         100 * DROP_DF["dur_s"].sum() / max(DEV.loc[IS_TRAIN, "dur_s"].sum(), 1e-9)))
print("data uji   : %d klip, %.2f jam" % (len(EVAL_DF), EVAL_DF["dur_s"].sum() / 3600))
assert len(TRAIN_DF) > 0, "tidak ada klip latih"
assert not set(TRAIN_DF["audio_filename"]) & set(EVAL_NAMES), "klip uji bocor ke data latih"
assert not set(TRAIN_DF["speaker"]) & set(EVAL_DF["speaker"]), "penutur uji juga ada di data latih"
assert (TRAIN_DF["transcript"].str.strip() != "").all(), "ada transkrip latih kosong"
assert float(TRAIN_DF["dur_s"].max()) <= MAX_TRAIN_S

rec("data", convo=_convo_stats, n_train=len(TRAIN_DF),
    train_hours=round(TRAIN_DF["dur_s"].sum() / 3600, 3), n_dropped_long=len(DROP_DF),
    dropped_hours=round(DROP_DF["dur_s"].sum() / 3600, 3), n_eval=len(EVAL_DF),
    eval_hours=round(EVAL_DF["dur_s"].sum() / 3600, 3),
    speakers_disjoint=not (set(TRAIN_DF["speaker"]) & set(EVAL_DF["speaker"])))

## 7 — Processor, dataset, collator, model, decoder kurva

Processor tahap 1 persis, dibuktikan sama dengan tokenizer checkpoint pada 372 transkrip. Collator NB 06 apa adanya (`-100`, prefiks dibuang). Decoder kurva greedy di dalam proses, konstanta `main.py`.

In [ ]:
# ============================================================
# CELL 7 - PROCESSOR / DATASET / COLLATOR / MODEL / DECODER KURVA
# ============================================================
from dataclasses import dataclass
from typing import Any, Dict, List
from torch.utils.data import Dataset as TorchDataset
from transformers import (WhisperProcessor, WhisperForConditionalGeneration, WhisperTokenizerFast,
                          Seq2SeqTrainer, TrainerCallback)
head("PROCESSOR")
DEVICE = "cuda"
HF_MODEL, LANG_CODE, LANG_NAME = "openai/whisper-large-v3-turbo", "jw", "javanese"

# Processor tahap 1 persis (NB 06 sel 6): label dibangun dengan prefiks yang sama.
PROCESSOR = WhisperProcessor.from_pretrained(HF_MODEL, language=LANG_NAME, task="transcribe")
TOK, FEX = PROCESSOR.tokenizer, PROCESSOR.feature_extractor
# Di transformers 4.57, language=/task= di WhisperProcessor.from_pretrained TIDAK sampai ke
# tokenizer (run 28 Sep: prefiks tanpa <|jw|>). Diset langsung, supaya label sama dengan tahap 1.
TOK.set_prefix_tokens(language=LANG_NAME, task="transcribe", predict_timestamps=False)
_prefix = TOK.convert_ids_to_tokens(TOK("halo rek", add_special_tokens=True).input_ids)[:4]
print("tokenizer    :", type(TOK).__name__, "| prefiks label:", _prefix)
# Prefiks tahap 1 persis (NB 06 E4-native, model_setup.label_prefix).
assert _prefix == ["<|startoftranscript|>", "<|%s|>" % LANG_CODE, "<|transcribe|>", "<|notimestamps|>"], \
    "prefiks label != tahap 1: %s" % _prefix

# Feature extractor dan tokenizer harus sama dengan milik checkpoint D, yang dipakai main.py.
FE_KEYS = ("feature_size", "sampling_rate", "hop_length", "chunk_length", "n_fft", "padding_value")
FE_DIFF = {k: (getattr(FEX, k, None), BASE_FE[k]) for k in FE_KEYS
           if k in BASE_FE and getattr(FEX, k, None) != BASE_FE[k]}
print("feature extractor vs checkpoint:", FE_DIFF or "sama")
assert not FE_DIFF, "feature extractor tahap 1 != checkpoint"
TOK_CK = WhisperTokenizerFast(tokenizer_file=str(BASE_DIR / "tokenizer.json"))
N_TOK_SAME = sum(TOK(t, add_special_tokens=False).input_ids == TOK_CK(t, add_special_tokens=False).input_ids
                 for t in DEV["transcript"])
print("token id 372 transkrip, tokenizer tahap 1 vs tokenizer.json checkpoint: %d/%d identik"
      % (N_TOK_SAME, len(DEV)))
assert N_TOK_SAME == len(DEV), "tokenizer tahap 1 dan checkpoint memecah teks secara berbeda"

# Panjang label (prefiks + teks + eos). generate() berhenti di GEN_MAX_LENGTH = 225 token, jadi
# klip pendek (satu jendela) yang labelnya lebih panjang dari itu PASTI terpotong saat decode.
GEN_MAX_LENGTH, NO_REPEAT_NGRAM, EVAL_BATCH = 225, 4, 8
LONG_S, CHUNK_S, OVERLAP_S, MIN_WINDOW_S, MAX_OVERLAP_WORDS = 29.8, 28.0, 2.0, 0.3, 8
MAX_LABEL_LEN = 448
DEV["label_len"] = [len(TOK(t.strip(), add_special_tokens=True).input_ids) for t in DEV["transcript"]]
_lab = {}
for c, g in DEV.groupby("convo_id"):
    s = g[g["dur_s"] <= LONG_S]
    _lab[c] = dict(max=int(g["label_len"].max()), p95=int(g["label_len"].quantile(0.95)),
                   n_short_over_gen_max=int((s["label_len"] > GEN_MAX_LENGTH).sum()),
                   n_over_448=int((g["label_len"] > MAX_LABEL_LEN).sum()))
    print("convo %s: label max %d, p95 %d | klip <= 29.8 s dengan label > %d token: %d | > 448: %d"
          % (c, _lab[c]["max"], _lab[c]["p95"], GEN_MAX_LENGTH, _lab[c]["n_short_over_gen_max"],
             _lab[c]["n_over_448"]))
rec("labels", per_convo=_lab, tokenizer_ids_identical=N_TOK_SAME, n_texts=len(DEV),
    tokenizer_class=type(TOK).__name__, prefix=[str(t) for t in _prefix])


class WhisperItems(TorchDataset):
    # NB 06 sel 6, disesuaikan: audio diambil dari AUDIO per nama klip, target = transcript resmi.
    def __init__(self, df):
        self.items, self.n_dropped = [], 0
        for r in df.itertuples():
            ids = TOK(r.transcript.strip(), add_special_tokens=True).input_ids
            if len(ids) > MAX_LABEL_LEN:
                self.n_dropped += 1
                continue
            self.items.append(dict(f=r.audio_filename, labels=ids))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        return {"audio": AUDIO[it["f"]], "labels": it["labels"]}


@dataclass
class WhisperCollator:
    # NB 06 sel 6 apa adanya: -100 di posisi padding, <|startoftranscript|> dibuang karena
    # Trainer menambahkannya sendiri saat menggeser label.
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = self.processor.feature_extractor(
            [f["audio"] for f in features], sampling_rate=SR, return_tensors="pt")
        label_batch = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        labels = label_batch["input_ids"].masked_fill(label_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

COLLATOR = WhisperCollator(PROCESSOR, TOK.convert_tokens_to_ids("<|startoftranscript|>"))


def apply_gen_config(model):
    # NB 06 sel 6: satu konfigurasi decode (greedy + guard), divalidasi sekarang, bukan
    # saat save_pretrained setelah training selesai.
    gc_ = model.generation_config
    gc_.language = LANG_CODE
    gc_.task = "transcribe"
    gc_.num_beams = 1
    gc_.do_sample = False
    for attr, default in (("temperature", 1.0), ("top_p", 1.0), ("top_k", 50)):
        if getattr(gc_, attr, None) is not None:
            setattr(gc_, attr, default)
    gc_.no_repeat_ngram_size = NO_REPEAT_NGRAM
    gc_.max_length = GEN_MAX_LENGTH
    gc_.use_cache = True
    try:
        gc_.forced_decoder_ids = None
    except Exception:
        pass
    try:
        gc_.validate(strict=True)
    except TypeError:
        pass
    return model


DERIVED = ("proj_out.weight", "encoder.embed_positions.weight")

def build_model():
    # Bobot D dalam fp32 (master weights), config dari checkpoint. Tensor yang hilang hanya
    # boleh yang memang diturunkan library (proj_out terikat embedding, posisi sinusoidal
    # encoder) - aturan yang sama dengan main.py.
    model, info = WhisperForConditionalGeneration.from_pretrained(
        str(BASE_DIR), dtype=torch.float32, output_loading_info=True)
    missing = info.get("missing_keys", [])
    bad = [k for k in missing if not k.endswith(DERIVED)] + list(info.get("mismatched_keys", []))
    assert not bad, "checkpoint tidak pas dengan model: %s" % bad[:5]
    model.config.use_cache = not GRAD_CHECKPOINT
    if SPEC_AUGMENT:
        model.config.apply_spec_augment = True
        model.config.mask_time_prob = 0.05
        model.config.mask_time_length = 10
        model.config.mask_feature_prob = 0.05
        model.config.mask_feature_length = 10
    if GRAD_CHECKPOINT:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    apply_gen_config(model)
    return model, dict(n_missing_derived=len(missing), n_unexpected=len(info.get("unexpected_keys", [])))


# ---- decoder kurva: greedy di dalam proses, konstanta main.py, fp16 autocast ----
def split_windows(y):
    step = int((CHUNK_S - OVERLAP_S) * SR)
    win = int(CHUNK_S * SR)
    parts = [y[i:i + win] for i in range(0, len(y), step)]
    return [p for p in parts if len(p) > int(MIN_WINDOW_S * SR)]

def join_overlapping(texts):
    joined = ""
    for t in texts:
        t = (t or "").strip()
        if not t:
            continue
        if not joined:
            joined = t
            continue
        a, b = joined.split(), t.split()
        k = 0
        for n in range(min(MAX_OVERLAP_WORDS, len(a), len(b)), 0, -1):
            if [w.lower() for w in a[-n:]] == [w.lower() for w in b[:n]]:
                k = n
                break
        joined = " ".join(a + b[k:])
    return joined.strip()

@torch.no_grad()
def _transcribe(model, arrays):
    feats = FEX(arrays, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        ids = model.generate(feats, task="transcribe", language=LANG_CODE, num_beams=1, do_sample=False,
                             no_repeat_ngram_size=NO_REPEAT_NGRAM, max_length=GEN_MAX_LENGTH,
                             return_timestamps=False, use_cache=True)
    return [t.strip() for t in TOK.batch_decode(ids, skip_special_tokens=True,
                                                clean_up_tokenization_spaces=CLEANUP)]

def decode_greedy(model, names):
    # Urutan klip dan komposisi batch seperti main.py (klip panjang langsung di-chunk).
    was = model.training
    model.eval()
    res, buf = {}, []
    def flush():
        for f, t in zip(buf, _transcribe(model, [AUDIO[x] for x in buf])):
            res[f] = t
        buf.clear()
    for f in names:
        y = AUDIO[f]
        if len(y) / SR > LONG_S:
            parts = split_windows(y)
            res[f] = join_overlapping(_transcribe(model, parts)) if parts else ""
        else:
            buf.append(f)
            if len(buf) == EVAL_BATCH:
                flush()
    if buf:
        flush()
    if was:
        model.train()
    return [res[f] for f in names]


class CurveCallback(TrainerCallback):
    # Kurva WER greedy di convo uji setelah tiap epoch. Tidak memilih apa pun: yang dikirim
    # tetap step terakhir. Error di sini tidak boleh menghentikan training.
    def __init__(self):
        self.rows = []
        self.disabled = not (CURVE and EVAL_NAMES)
    def measure(self, model, epoch, step):
        t0 = time.time()
        hyps = decode_greedy(model, EVAL_NAMES)
        E = clip_errors([REF[f] for f in EVAL_NAMES], hyps)
        n_hyp = sum(len(normalize_text(h).split()) for h in hyps)
        row = dict(epoch=round(float(epoch), 3), step=int(step), wer=round(E[:, 0].sum() / max(E[:, 1].sum(), 1), 5),
                   n_err=int(E[:, 0].sum()), n_tok=int(E[:, 1].sum()),
                   len_ratio=round(n_hyp / max(E[:, 1].sum(), 1), 4), seconds=round(time.time() - t0, 1))
        self.rows.append(row)
        print("  [KURVA convo %s, greedy] epoch %-5s step %-4d WER %.4f (%d/%d)  rasio panjang %.3f  %.0f s"
              % ("+".join(EVAL_CONVOS), row["epoch"], row["step"], row["wer"], row["n_err"], row["n_tok"],
                 row["len_ratio"], row["seconds"]))
    def on_epoch_end(self, args, state, control, model=None, **kw):
        if self.disabled or model is None:
            return
        try:
            self.measure(model, state.epoch or 0.0, state.global_step)
        except Exception as e:
            self.disabled = True
            print("  [kurva gagal, dimatikan untuk sisa run: %r]" % (e,))

print("collator: decoder_start_token_id", COLLATOR.decoder_start_token_id)

## 8 — Training tahap 2

Step terakhir yang dikirim. Kurva di convo uji hanya observasi. Checkpoint disimpan sekali sebagai uji sebelum training, supaya kegagalan simpan di transformers 4.x muncul sebelum GPU dipakai.

In [ ]:
# ============================================================
# CELL 8 - TRAINING TAHAP 2
# ============================================================
head("TRAINING")
_TA_ALIASES = {"eval_strategy": ["eval_strategy", "evaluation_strategy"]}

def make_training_args(**kw):
    # NB 06 sel 9: argumen yang tidak dikenal versi ini dilaporkan dan dibuang, bukan crash.
    sig = set(inspect.signature(Seq2SeqTrainingArguments.__init__).parameters)
    fixed, dropped = {}, []
    for k, v in kw.items():
        if k in sig:
            fixed[k] = v
            continue
        alt = next((a for a in _TA_ALIASES.get(k, []) if a in sig), None)
        if alt:
            fixed[alt] = v
        else:
            dropped.append(k)
    if dropped:
        print("  [training args] tidak dikenal transformers %s, dibuang: %s" % (transformers.__version__, dropped))
    return Seq2SeqTrainingArguments(**fixed)

TRAINER = MODEL = None
CURVE_CB = CurveCallback()
TRAIN_INFO = {}
if RUN_MODE != "train":
    print("RUN_MODE=%r - training dilewati; sel 9 mengunduh checkpoint S dari HF." % RUN_MODE)
else:
    train_ds = WhisperItems(TRAIN_DF)
    n_micro = math.ceil(len(train_ds) / PER_DEVICE_BATCH)
    STEPS_EXPECTED = math.ceil(n_micro / GRAD_ACCUM) * NUM_EPOCHS
    print("dataset latih : %d klip (%d dibuang karena label > %d token)"
          % (len(train_ds), train_ds.n_dropped, MAX_LABEL_LEN))
    print("rencana       : %d epoch x ~%d step = ~%d step optimizer, LR %.1e, warmup %.0f%%"
          % (NUM_EPOCHS, math.ceil(n_micro / GRAD_ACCUM), STEPS_EXPECTED, LEARNING_RATE, 100 * WARMUP_RATIO))

    # Collator dibuktikan, bukan diasumsikan: prefiks dibuang, padding jadi -100.
    _b = COLLATOR([train_ds[i] for i in range(min(4, len(train_ds)))])
    _lb = _b["labels"]
    print("batch contoh  : fitur %s, label %s, %.1f%% posisi -100, token pertama %s"
          % (tuple(_b["input_features"].shape), tuple(_lb.shape),
             100.0 * float((_lb == -100).sum()) / _lb.numel(),
             TOK.convert_ids_to_tokens([int(_lb[0, 0])])))
    assert (_lb[:, 0] != COLLATOR.decoder_start_token_id).all(), "<|startoftranscript|> tidak dibuang"
    assert tuple(_b["input_features"].shape[1:]) == (128, 3000)

    t0 = time.time()
    MODEL, LOAD_INFO = build_model()
    MODEL.to(DEVICE)
    print("model dimuat  : %.0f s | tensor diturunkan %d, tak terduga %d | %.0f M parameter"
          % (time.time() - t0, LOAD_INFO["n_missing_derived"], LOAD_INFO["n_unexpected"],
             sum(p.numel() for p in MODEL.parameters()) / 1e6))

    # Uji simpan SEBELUM training: kalau save_pretrained transformers 4.x menolak config
    # checkpoint 5.0.0, gagalnya sekarang, bukan setelah 10 menit GPU.
    _probe_dir = TEMP / "save_probe"
    shutil.rmtree(_probe_dir, ignore_errors=True)
    t0 = time.time()
    MODEL.save_pretrained(str(_probe_dir), safe_serialization=True)
    _probe_files = sorted(p.name for p in _probe_dir.iterdir())
    print("uji simpan    : OK, %.0f s, %s" % (time.time() - t0, _probe_files))
    shutil.rmtree(_probe_dir, ignore_errors=True)

    targs = make_training_args(
        output_dir=str(CKPT_DIR / "run"), per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO, lr_scheduler_type="linear", num_train_epochs=NUM_EPOCHS,
        fp16=True, gradient_checkpointing=GRAD_CHECKPOINT,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="no", save_strategy="no", logging_steps=5,
        remove_unused_columns=False, label_names=["labels"], dataloader_num_workers=0,
        optim=OPTIM, seed=SEED, data_seed=SEED, report_to=[], disable_tqdm=False)

    # Epoch 0 = D sendiri (greedy, fp32 + autocast, batch convo uji saja). Dekat dengan N di
    # convo uji (N_REF), tapi tidak harus sama persis: jalur numeriknya lain.
    if not CURVE_CB.disabled:
        CURVE_CB.measure(MODEL, 0.0, 0)
        MODEL.train()

    TRAINER = Seq2SeqTrainer(model=MODEL, args=targs, train_dataset=train_ds,
                             data_collator=COLLATOR, processing_class=PROCESSOR, callbacks=[CURVE_CB])
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    TRAIN_RESULT = TRAINER.train()
    train_min = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    rows = [dict(step=h.get("step"), epoch=h.get("epoch"), loss=h.get("loss"),
                 learning_rate=h.get("learning_rate"), grad_norm=h.get("grad_norm"))
            for h in TRAINER.state.log_history if "loss" in h]
    TRAIN_LOG = pd.DataFrame(rows)
    TRAIN_LOG.to_csv(out("train_log.csv"), index=False)
    print("\nlog training (tiap 5 step):")
    print(TRAIN_LOG.to_string(index=False) if len(TRAIN_LOG) else "  (kosong)")
    losses = [float(x) for x in TRAIN_LOG.get("loss", []) if x is not None]
    CURVE_DF = pd.DataFrame(CURVE_CB.rows)
    if len(CURVE_DF):
        CURVE_DF.to_csv(out("curve_eval_greedy.csv"), index=False)
        print("\nkurva greedy di convo uji (observasi, bukan seleksi):")
        print(CURVE_DF.to_string(index=False))
        if EVAL_CONVOS == ("2",) and N_REF:
            print("konteks: N (E4-native greedy lewat main.py) di convo 2 = %d error / %d token"
                  % N_REF["2"])

    TRAIN_INFO = dict(
        n_items=len(train_ds), n_dropped_label=train_ds.n_dropped, steps_expected=STEPS_EXPECTED,
        global_step=int(TRAINER.state.global_step), epoch=round(float(TRAINER.state.epoch or 0), 3),
        minutes=round(train_min, 2), peak_vram_gb=round(peak_gb, 2),
        loss_first=(round(losses[0], 4) if losses else None), loss_last=(round(losses[-1], 4) if losses else None),
        loss_finite=bool(losses) and all(math.isfinite(x) for x in losses),
        train_loss=round(float(TRAIN_RESULT.training_loss), 4), optimizer=str(getattr(targs.optim, "value", targs.optim)),
        load_missing_derived=LOAD_INFO["n_missing_derived"], load_unexpected=LOAD_INFO["n_unexpected"],
        save_probe_files=_probe_files, curve=CURVE_CB.rows)
    print("\ntraining: %d step (rencana ~%d), epoch %.2f, %.1f menit, VRAM puncak %.1f GB, loss %s -> %s"
          % (TRAIN_INFO["global_step"], STEPS_EXPECTED, TRAIN_INFO["epoch"], train_min, peak_gb,
             TRAIN_INFO["loss_first"], TRAIN_INFO["loss_last"]))
rec("training", **TRAIN_INFO)

## 9 — Checkpoint S: simpan, periksa, push

S = file non-bobot D + `model.safetensors` baru, jadi NB 08 bisa memaketkannya tanpa jalur muat baru. Dipush sebelum evaluasi.

In [ ]:
# ============================================================
# CELL 9 - CHECKPOINT S: SIMPAN, PERIKSA, PUSH (atau unduh di RUN_MODE="eval")
# ============================================================
head("CHECKPOINT S")
SHIP_DIR = TEMP / "ckpt_S" / "model"
# Checkpoint S = file non-bobot D byte demi byte + model.safetensors baru. Tata letak
# (config/processor/tokenizer/generation) itu yang sudah terbukti jalan di container untuk
# D, jadi NB 08 bisa memaketkan S tanpa jalur muat baru.
COPY_FROM_BASE = [n for n in sorted(BASE_FILES) if n != "model.safetensors"]
CKPT_INFO, S_REVISION = {}, None

if RUN_MODE == "train":
    raw = CKPT_DIR / "saved"
    shutil.rmtree(raw, ignore_errors=True)
    TRAINER.save_model(str(raw))
    raw_files = sorted(p.name for p in raw.iterdir())
    print("disimpan transformers %s: %s" % (transformers.__version__, raw_files))
    assert (raw / "model.safetensors").exists(), "model.safetensors tidak ditulis"
    assert not any(n.endswith(".index.json") for n in raw_files), "bobot tersharding - main.py butuh satu file"

    # Arsitektur harus sama; config.json yang dikirim tetap milik D.
    cfg_b, cfg_n = _j(BASE_DIR / "config.json"), _j(raw / "config.json")
    ARCH = ("d_model", "encoder_layers", "decoder_layers", "encoder_attention_heads",
            "decoder_attention_heads", "encoder_ffn_dim", "decoder_ffn_dim", "vocab_size",
            "num_mel_bins", "max_source_positions", "max_target_positions")
    arch_diff = {k: (cfg_b.get(k), cfg_n.get(k)) for k in ARCH if cfg_b.get(k) != cfg_n.get(k)}
    other_diff = sorted(k for k in set(cfg_b) | set(cfg_n) if cfg_b.get(k) != cfg_n.get(k))
    print("config arsitektur vs D:", arch_diff or "sama", "| kunci lain yang beda:", other_diff)
    assert not arch_diff, "arsitektur hasil latih != D"

    if SHIP_DIR.parent.exists():
        shutil.rmtree(SHIP_DIR.parent)
    SHIP_DIR.mkdir(parents=True)
    for n in COPY_FROM_BASE:
        shutil.copy2(BASE_DIR / n, SHIP_DIR / n)
    shutil.move(str(raw / "model.safetensors"), str(SHIP_DIR / "model.safetensors"))
    shutil.rmtree(raw, ignore_errors=True)
    shutil.rmtree(CKPT_DIR / "run", ignore_errors=True)

    # GPU dikosongkan untuk main.py (subprocess) di sel 10.
    try:
        TRAINER.accelerator.free_memory()
    except Exception:
        pass
    MODEL.to("cpu")
    TRAINER = MODEL = None
    gc.collect(); torch.cuda.empty_cache()
    print("GPU setelah dikosongkan: %.2f GB teralokasi" % (torch.cuda.memory_allocated() / 1e9))
    CKPT_INFO.update(arch_same=not arch_diff, config_keys_differing=other_diff, saved_files=raw_files)
else:
    t0 = time.time()
    _rev = EVAL_REVISION or "main"
    fetch_checkpoint(HF_REPO, _rev, SHIP_DIR)
    S_REVISION = EVAL_REVISION or API.model_info(HF_REPO, revision="main").sha
    print("diunduh %s @ %s dalam %.0f s" % (HF_REPO, S_REVISION, time.time() - t0))

ST = SHIP_DIR / "model.safetensors"
S_SHA16 = sha16(ST)
if RUN_MODE == "eval" and EVAL_SHA16:
    assert S_SHA16 == EVAL_SHA16, "checkpoint S yang diunduh (%s) != EVAL_SHA16 (%s)" % (S_SHA16, EVAL_SHA16)

# Nama tensor: hanya boleh beda di tensor yang diturunkan library (aturan main.py).
with safe_open(str(ST), "pt") as f:
    S_KEYS = set(f.keys())
KEY_DIFF = sorted(k for k in (S_KEYS ^ BASE_KEYS) if not k.endswith(DERIVED))
print("tensor: S %d, D %d | beda di luar tensor turunan: %s" % (len(S_KEYS), len(BASE_KEYS), KEY_DIFF[:5] or "tidak ada"))

# Seberapa jauh bobot bergeser dari D: ||S - D|| / ||D|| per bagian. 0 = tidak terlatih;
# puluhan persen = rusak. Dihitung per tensor di CPU.
_mv = collections.defaultdict(lambda: [0.0, 0.0])
n_changed, n_common, s_dtypes = 0, 0, collections.Counter()
with safe_open(str(ST), "pt") as fs, safe_open(str(BASE_DIR / "model.safetensors"), "pt") as fb:
    for k in sorted(S_KEYS & BASE_KEYS):
        ts = fs.get_tensor(k)
        s_dtypes[str(ts.dtype).replace("torch.", "")] += 1
        if not ts.is_floating_point():
            continue
        a, b = fb.get_tensor(k).double(), ts.double()
        d2, w2 = float((b - a).pow(2).sum()), float(a.pow(2).sum())
        grp = "encoder" if ".encoder." in k else ("decoder" if ".decoder." in k else "lain")
        _mv[grp][0] += d2; _mv[grp][1] += w2
        _mv["semua"][0] += d2; _mv["semua"][1] += w2
        n_changed += int(d2 > 0); n_common += 1
        del a, b, ts
REL_DELTA = {g: round(math.sqrt(v[0] / max(v[1], 1e-30)), 6) for g, v in _mv.items()}
print("pergeseran bobot relatif:", REL_DELTA, "| tensor berubah %d/%d | dtype %s"
      % (n_changed, n_common, dict(s_dtypes)))

S_FILES = {p.name: sha16(p) for p in sorted(SHIP_DIR.iterdir()) if p.is_file()}
same_nonweight = all(S_FILES.get(n) == BASE_FILES[n] for n in COPY_FROM_BASE)
print("file non-bobot identik dengan D:", same_nonweight)

if RUN_MODE == "train":
    if HF_ENABLED:
        t0 = time.time()
        _info = API.upload_folder(folder_path=str(SHIP_DIR), repo_id=HF_REPO, repo_type="model",
                                  revision="main",
                                  commit_message="%s: tahap 2 dari %s @%s, convo %s, %d epoch, LR %g, step %s"
                                  % (RUN_ID, BASE["name"], BASE["rev"][:8], "+".join(TRAIN_CONVOS), NUM_EPOCHS,
                                     LEARNING_RATE, TRAIN_INFO.get("global_step")))
        S_REVISION = getattr(_info, "oid", None) or getattr(_info, "commit_id", None) or str(_info)
        print("push -> %s @ %s (%.0f s)" % (HF_REPO, S_REVISION, time.time() - t0))
    else:
        print("!!! Hub mati: bobot S TIDAK dipush dan hilang bersama sesi.")

CKPT_MANIFEST = dict(
    run_id=RUN_ID, base_repo=BASE["repo"], base_revision=BASE["rev"], base_sha16=BASE["sha16"],
    hf_repo=HF_REPO, hf_revision=S_REVISION, hf_branch="main", private=True,
    safetensors_sha16=S_SHA16, files=S_FILES, nonweight_files_from_base=same_nonweight,
    train_convos=list(TRAIN_CONVOS), eval_convos=list(EVAL_CONVOS), max_train_s=MAX_TRAIN_S,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO,
    effective_batch=PER_DEVICE_BATCH * GRAD_ACCUM, optim=OPTIM, seed=SEED,
    global_step=TRAIN_INFO.get("global_step"), trained_with_transformers=transformers.__version__,
    checkpoint_policy="final")
out("checkpoint_manifest.json").write_text(json.dumps(jsonable(CKPT_MANIFEST), indent=1, sort_keys=True),
                                           encoding="utf-8")
if RUN_MODE == "train" and HF_ENABLED:
    try:
        API.upload_file(path_or_fileobj=str(out("checkpoint_manifest.json")),
                        path_in_repo="checkpoint_manifest.json", repo_id=HF_REPO, repo_type="model")
    except Exception as e:
        print("  !! upload manifest gagal (bobot sudah aman): %r" % (e,))
print("\nS = %s @ %s | model.safetensors sha16 %s" % (HF_REPO, S_REVISION, S_SHA16))
print("    RUN_MODE='eval' memakai dua angka ini kalau evaluasi perlu diulang.")
CKPT_INFO.update(hf_repo=HF_REPO, hf_revision=S_REVISION, sha16=S_SHA16, key_diff=KEY_DIFF[:10],
                 rel_delta=REL_DELTA, n_tensors_changed=n_changed, n_tensors_common=n_common,
                 dtypes=dict(s_dtypes), nonweight_same_as_base=same_nonweight, files=S_FILES)
rec("checkpoint", **CKPT_INFO)

## 10 — Evaluasi lewat `main.py` (container tiruan)

Subprocess dengan cwd dan env offline seperti entrypoint resmi, 372 klip dalam urutan NB 08.

In [ ]:
# ============================================================
# CELL 10 - EVALUASI LEWAT main.py, SEPERTI ENTRYPOINT CONTAINER
# ============================================================
head("EVALUASI LEWAT main.py (beam %d)" % NUM_BEAMS)
# Kedua arm di-decode oleh main.py yang sama, di 372 klip dalam urutan metadata - urutan
# NB 08, jadi komposisi batch D di sini sama persis dengan run yang menghasilkan 3.965 error.
# Convo latih ikut di-decode: angkanya "fit data latih", bukan hasil.
#   D = kontrol: checkpoint awal. Harus menyamai D_REF (NB 08). Hanya kalau ada convo uji.
#   S = checkpoint tahap 2.
ARMS = {}
if EVAL_NAMES:
    ARMS["D"] = dict(model=BASE_DIR, what="D: titik awal %s, beam %d (kontrol, harus = D_REF)"
                     % (BASE["name"], NUM_BEAMS))
ARMS["S"] = dict(model=SHIP_DIR, what="S: tahap 2 (%s), beam %d" % (RUN_NOTE, NUM_BEAMS))
ORDER = list(DEV["audio_filename"])
LANG_OF = dict(zip(DEV["audio_filename"], DEV["language"]))
ENV = {k: v for k, v in os.environ.items() if k != "LIT_DEBUG"}
ENV.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")
CE_ROOT = TEMP / "code_execution"

def build_container(root, model_dir):
    # data/clips + data/test_metadata.csv + src/{main.py, postprocess.py, model -> bobot} + submission/.
    # Klip dan bobot di-symlink: tidak ada salinan 3 GB.
    if root.exists():
        shutil.rmtree(root)
    (root / "data" / "clips").mkdir(parents=True)
    (root / "submission").mkdir(parents=True)
    (root / "src").mkdir(parents=True)
    for f in ORDER:
        os.symlink(DEV_CLIPS / f, root / "data" / "clips" / f)
    pd.DataFrame(dict(audio_filename=ORDER,
                      file_duration_seconds=[int(round(DUR[f])) for f in ORDER],
                      language=[LANG_OF[f] for f in ORDER])).to_csv(root / "data" / "test_metadata.csv", index=False)
    (root / "src" / "main.py").write_text(MAIN_EVAL, encoding="utf-8")
    (root / "src" / "postprocess.py").write_text(SRC_POST, encoding="utf-8")
    (root / "src" / "model").symlink_to(model_dir, target_is_directory=True)

def load_times(stdout):
    m = re.search(r"import ([\d.]+)s \| model load ([\d.]+)s", stdout)
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

def decode_paths(stdout):
    m = re.search(r"short (\d+) \| chunked (\d+) \((\d+) windows\) \| failed (\d+) \| empty (\d+)", stdout)
    return dict(zip(("short", "chunked", "windows", "failed", "empty"), map(int, m.groups()))) if m else {}

RUNS = {}
for arm, a in ARMS.items():
    print("\n--- arm %s: %s ---" % (arm, a["what"]))
    root = CE_ROOT / arm
    build_container(root, a["model"])
    t0 = time.time()
    proc = subprocess.run([sys.executable, "src/main.py"], cwd=str(root), env=ENV,
                          capture_output=True, text=True, timeout=7200)
    wall = time.time() - t0
    log_all = proc.stdout + proc.stderr
    print(proc.stdout[-1200:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-4000:])
        rec("run." + arm, exit_code=proc.returncode, stderr_tail=proc.stderr[-600:])
        dump_handoff()
        raise RuntimeError("main.py gagal di arm %s - kirim STDERR dan blok JSON di atas ke Claude" % arm)
    PRED = pd.read_csv(root / "submission" / "submission.csv", keep_default_na=False)
    assert list(PRED.columns) == ["audio_filename", "transcript"], "header submission.csv salah"
    assert list(PRED["audio_filename"]) == ORDER, "daftar/urutan klip submission.csv salah"
    shutil.copy(root / "submission" / "submission.csv", out("submission_dev_%s.csv" % arm))
    imp_s, load_s = load_times(proc.stdout)
    paths = decode_paths(proc.stdout)
    leak = sum(1 for f in ORDER if f.rsplit(".", 1)[0] in log_all)
    n_empty = int((PRED["transcript"].str.strip() == "").sum())
    print("wall %.0f s (%.2f s/klip) | import %s s, load %s s | jalur %s | kosong %d | nama klip di log %d"
          % (wall, wall / len(ORDER), imp_s, load_s, paths, n_empty, leak))
    RUNS[arm] = dict(hyp=dict(zip(PRED["audio_filename"], PRED["transcript"])), paths=paths)
    rec("run." + arm, exit_code=proc.returncode, wall_s=round(wall, 1), s_per_clip=round(wall / len(ORDER), 4),
        import_s=imp_s, load_s=load_s, paths=paths, paths_match=(paths == dict(REF_PATHS, failed=0, empty=0)),
        n_empty=n_empty, leak_names=leak, n_log_lines=len(log_all.splitlines()))
    shutil.rmtree(root, ignore_errors=True)

## 11 — Skor, paritas kontrol D, dan S − D berpasangan

Verdict mengikuti aturan yang ditulis di atas sebelum run.

In [ ]:
# ============================================================
# CELL 11 - SKOR RESMI, PARITAS D, DAN S - D BERPASANGAN
# ============================================================
head("SKOR")
CONVO = DEV["convo_id"].to_numpy()
LANGV = DEV["language"].to_numpy()
SPKV  = DEV["speaker"].to_numpy()
REFS_ORDER = [REF[f] for f in ORDER]
ERRS, ERRS_LN = {}, {}
for arm in RUNS:
    hyps = [RUNS[arm]["hyp"][f] for f in ORDER]
    ERRS[arm] = clip_errors(REFS_ORDER, hyps)
    ERRS_LN[arm] = clip_errors_ln(REFS_ORDER, hyps)

MASKS = {"all": np.ones(len(ORDER), bool)}
for c in sorted(set(CONVO)):
    MASKS[c] = CONVO == c

def wer_of(E, m):
    e, n = int(E[m, 0].sum()), int(E[m, 1].sum())
    return e / max(n, 1), e, n

def role(k):
    return ("semua" if k == "all" else "convo %s (%s)" % (k, "LATIH - fit, bukan hasil" if k in TRAIN_CONVOS
            else "UJI" if k in EVAL_CONVOS else "-"))

SCORES = {}
for arm in RUNS:
    SCORES[arm] = {}
    print("\narm %s:" % arm)
    for k, m in MASKS.items():
        w, e, n = wer_of(ERRS[arm], m)
        wl, _, _ = wer_of(ERRS_LN[arm], m)
        SCORES[arm][k] = dict(wer=round(w, 6), n_err=e, n_tok=n, wer_lower_nopunct=round(wl, 6))
        print("   %-34s WER %.4f  (%4d / %5d)   lower+tanpa tanda baca %.4f" % (role(k), w, e, n, wl))

# Kontrol D: main.py yang sama, bobot yang sama, urutan yang sama dengan NB 08 -> errornya
# harus sama. Kalau tidak, lingkungan decode di sini bergeser dan angka S tidak bisa
# dibandingkan dengan submission D; perbandingan berpasangan S - D di bawah tetap sah
# karena keduanya di-decode di sesi ini.
PARITY = {}
if "D" in RUNS:
    print("\nkontrol D vs NB 08 (normalizer resmi):")
    for k, (e0, n0) in D_REF.items():
        got = SCORES["D"].get(k)
        if got is None:
            continue
        d = got["wer"] - e0 / n0
        PARITY[k] = dict(n_err=got["n_err"], ref_err=e0, delta_err=got["n_err"] - e0, delta_wer=round(d, 6),
                         ok=bool(got["n_tok"] == n0 and abs(d) <= PARITY_TOL))
        print("   %-8s %4d error vs %4d (NB 08) -> %+d, %+.4f %s"
              % (k, got["n_err"], e0, got["n_err"] - e0, d, "OK" if PARITY[k]["ok"] else "!!! BERGESER"))

COMPARE, VERDICT = {}, "n/a (tidak ada convo uji)"
if "D" in RUNS and EVAL_NAMES:
    EV = np.isin(CONVO, EVAL_CONVOS)
    TR = np.isin(CONVO, TRAIN_CONVOS)
    print("\nS - D berpasangan, bootstrap per klip %d resample. Negatif = S lebih baik." % N_BOOT)
    def show(label, res):
        print("   %-40s %+.4f  CI95 [%+.4f, %+.4f]  p(S lebih baik) %.3f"
              % (label, res["delta"], res["ci95"][0], res["ci95"][1], res["p_better"]))
    COMPARE["eval"] = paired(ERRS["S"][EV], ERRS["D"][EV])
    show("UJI convo %s (%d klip)" % ("+".join(EVAL_CONVOS), int(EV.sum())), COMPARE["eval"])
    COMPARE["eval_lower_nopunct"] = paired(ERRS_LN["S"][EV], ERRS_LN["D"][EV])
    show("  ruang lower + tanpa tanda baca", COMPARE["eval_lower_nopunct"])
    for lg in sorted(set(LANGV[EV])):
        m = EV & (LANGV == lg)
        COMPARE["eval_lang_" + lg] = dict(paired(ERRS["S"][m], ERRS["D"][m]), n_clips=int(m.sum()),
                                          n_tok=int(ERRS["D"][m, 1].sum()))
        show("  bahasa %s (%d klip)" % (lg, int(m.sum())), COMPARE["eval_lang_" + lg])
    # Penutur disebut berdasarkan urutan jumlah token, bukan id-nya.
    _spk = sorted(set(SPKV[EV]), key=lambda s: -int(ERRS["D"][EV & (SPKV == s), 1].sum()))
    for i, s in enumerate(_spk, 1):
        m = EV & (SPKV == s)
        COMPARE["eval_speaker_%d" % i] = dict(paired(ERRS["S"][m], ERRS["D"][m]), n_clips=int(m.sum()),
                                              n_tok=int(ERRS["D"][m, 1].sum()))
        show("  penutur uji #%d (%d klip)" % (i, int(m.sum())), COMPARE["eval_speaker_%d" % i])
    COMPARE["train_fit"] = paired(ERRS["S"][TR], ERRS["D"][TR])
    show("LATIH convo %s (fit, BUKAN hasil)" % "+".join(TRAIN_CONVOS), COMPARE["train_fit"])

    # Seberapa banyak yang berubah, dan apakah S jadi lebih pendek/panjang.
    _ev_names = [f for f, m in zip(ORDER, EV) if m]
    n_same = sum(normalize_text(RUNS["S"]["hyp"][f]) == normalize_text(RUNS["D"]["hyp"][f]) for f in _ev_names)
    n_ref = int(ERRS["D"][EV, 1].sum())
    len_ratio = {arm: round(sum(len(normalize_text(RUNS[arm]["hyp"][f]).split()) for f in _ev_names) / max(n_ref, 1), 4)
                 for arm in ("D", "S")}
    print("\ntranskrip uji identik S vs D: %d/%d | rasio panjang hyp/ref: D %.3f, S %.3f"
          % (n_same, len(_ev_names), len_ratio["D"], len_ratio["S"]))

    lo, hi = COMPARE["eval"]["ci95"]
    VERDICT = ("JELAS LEBIH BAIK" if hi < 0 else "LEBIH BURUK" if lo > 0 else "TIDAK JELAS")
    COMPARE.update(n_identical_eval=n_same, n_eval=len(_ev_names), len_ratio_eval=len_ratio)
    print("\n>>> VERDICT (aturan dari markdown atas: CI95 S - D di convo uji seluruhnya < 0): %s" % VERDICT)
    print("    D di convo uji %.4f -> S %.4f. Konteks, bukan ramalan: D di test %.4f, #1 leaderboard %.4f."
          % (SCORES["D"][EVAL_CONVOS[0]]["wer"] if len(EVAL_CONVOS) == 1 else float("nan"),
             SCORES["S"][EVAL_CONVOS[0]]["wer"] if len(EVAL_CONVOS) == 1 else float("nan"),
             KNOWN["D_test"], KNOWN["leaderboard_1"]))
    print("    Convo uji cuma 2 penutur: CI per klip meremehkan variasi antar penutur.")
    if globals().get("S_C5_REF") and EVAL_CONVOS == ("2",):
        e0, n0 = S_C5_REF["2"]
        print("    PEMBANDING SEBENARNYA: S-c5 di convo 2 = %d / %d = %.4f -> S run ini %d / %d = %.4f (%+d error)."
              % (e0, n0, e0 / n0, SCORES["S"]["2"]["n_err"], SCORES["S"]["2"]["n_tok"],
                 SCORES["S"]["2"]["wer"], SCORES["S"]["2"]["n_err"] - e0))
        print("    CI berpasangan S - S-c5 dihitung di laptop dari dua submission_dev_S.csv.")
else:
    print("\nTidak ada convo uji: tidak ada perbandingan. Angka S di atas adalah fit data latih,")
    print("dan berguna sebagai REFS paritas untuk NB 08 (dihitung dari submission_dev_S.csv).")

rec("score", per_arm=SCORES, parity_D=PARITY)
rec("compare", verdict=VERDICT, **COMPARE)

## 12 — Self-validation dan handoff

In [ ]:
# ============================================================
# CELL 12 - SELF-VALIDATION, DIGEST, HANDOFF
# Salin SELURUH blok antara penanda BEGIN/END ke chat.
# ============================================================
head("SELF-VALIDATION")
CHECKS = []
def check(name, cond, detail=""):
    CHECKS.append(dict(check=name, passed=bool(cond), detail=str(detail)[:180]))
    print("  [%s] %-62s %s" % ("PASS" if cond else "FAIL", name, detail))

check("transformers = versi container (<5)", MAJOR < 5, transformers.__version__)
check("bobot awal = D (sha16 %s)" % BASE["sha16"], FINDINGS["base"]["sha_ok"], FINDINGS["base"]["sha16"])
check("main.py evaluasi = main.py D yang dikirim", FINDINGS["source"]["same_as_shipped_D"] or NUM_BEAMS != 5,
      FINDINGS["source"]["main_eval_sha16"])
check("convo latih dan uji terpisah, penutur juga", FINDINGS["data"]["speakers_disjoint"],
      "latih %s, uji %s" % (list(TRAIN_CONVOS), list(EVAL_CONVOS)))
check("tidak ada klip latih > %.0f s" % MAX_TRAIN_S, float(TRAIN_DF["dur_s"].max()) <= MAX_TRAIN_S,
      "%d klip dibuang" % len(DROP_DF))
check("tokenizer tahap 1 = tokenizer.json checkpoint", N_TOK_SAME == len(DEV), "%d/%d" % (N_TOK_SAME, len(DEV)))
if RUN_MODE == "train":
    T = FINDINGS["training"]
    check("training sampai akhir jadwal (step terakhir dikirim)", T["epoch"] >= NUM_EPOCHS - 0.01,
          "step %d, epoch %.2f (rencana ~%d)" % (T["global_step"], T["epoch"], T["steps_expected"]))
    check("loss hingga akhir terhingga", T["loss_finite"], "%s -> %s" % (T["loss_first"], T["loss_last"]))
    check("S dipush ke repo HF private dengan revisi", bool(FINDINGS["checkpoint"].get("hf_revision")),
          FINDINGS["checkpoint"].get("hf_revision"))
CK = FINDINGS["checkpoint"]
check("nama tensor S = D (di luar tensor turunan)", not CK["key_diff"], CK["key_diff"][:3] or "sama")
check("bobot bergeser, tapi wajar (0 < relatif < 0,05)", 0 < CK["rel_delta"].get("semua", 0) < 0.05,
      CK["rel_delta"])
check("file non-bobot S identik dengan D", CK["nonweight_same_as_base"])
for arm in RUNS:
    R = FINDINGS["run." + arm]
    check("arm %s: main.py selesai, tanpa transkrip kosong" % arm, R["exit_code"] == 0 and R["n_empty"] == 0,
          "%d kosong" % R["n_empty"])
    check("arm %s: jalur decode = 328 / 44 / 88" % arm, R["paths_match"], R["paths"])
    check("arm %s: log tanpa nama klip, <= 20 baris" % arm, R["leak_names"] == 0 and R["n_log_lines"] <= 20,
          "%d nama, %d baris" % (R["leak_names"], R["n_log_lines"]))
if "D" in RUNS:
    for k, p in PARITY.items():
        check("kontrol D = NB 08 di %s (toleransi %.3f)" % ("semua klip" if k == "all" else "convo " + k, PARITY_TOL),
              p["ok"], "%d vs %d error (%+d)" % (p["n_err"], p["ref_err"], p["delta_err"]))
_dl = [p for p in WORKING.rglob("*") if p.is_file()]
_w = [p for p in _dl if p.suffix in (".safetensors", ".bin", ".pt", ".pth")]
check("tidak ada bobot di /kaggle/working", not _w, "%d file bobot" % len(_w))
check("unduhan kecil (bobot di HF)", sum(p.stat().st_size for p in _dl) < 200e6,
      "%.1f MB" % (sum(p.stat().st_size for p in _dl) / 1e6))

n_pass = sum(c["passed"] for c in CHECKS)
print("\n%d/%d checks passed" % (n_pass, len(CHECKS)))
rec("validation", checks=CHECKS, n_passed=n_pass, n_total=len(CHECKS))

FINDINGS["_meta"] = dict(notebook_id=NOTEBOOK_ID, run_id=RUN_ID, run_mode=RUN_MODE,
                         generated_for="handoff ke Claude - agregat saja, jangan diedit",
                         artifacts=sorted(p.name for p in OUT_DIR.iterdir()) + ["dataset_findings.json"])
out("dataset_findings.json").write_text(json.dumps(jsonable(FINDINGS), ensure_ascii=False, indent=1,
                                                   sort_keys=True), encoding="utf-8")
sys.stdout.flush()
dump_handoff()
try:
    _zip = shutil.make_archive(str(WORKING / (RUN_ID + "-lite")), "zip",
                               root_dir=str(OUT_DIR.parent), base_dir=RUN_ID)
    print("\nunduhan satu file: %s (%.1f KB)" % (_zip, Path(_zip).stat().st_size / 1024))
except Exception as e:
    print("zip lite gagal: %r" % (e,))

head("LANGKAH BERIKUTNYA")
print("1. Salin blok CLAUDE-HANDOFF-JSON ke chat, dan unduh findings/%s/ ke repo." % RUN_ID)
if n_pass < len(CHECKS):
    print("2. Ada check FAIL. Baca dulu sebelum mempercayai angka apa pun di atas.")
elif EVAL_NAMES:
    print("2. Verdict: %s." % VERDICT)
    if VERDICT == "JELAS LEBIH BAIK":
        print("   Varian final: RUN_NOTE='all', TRAIN_CONVOS=('2', '5'), EVAL_CONVOS=(), resep sama,")
        print("   Run All di sesi baru. Lalu NB 08: entri CHECKPOINTS baru + arm beam 5, dan slot test")
        print("   melawan D 0,2901. Tidak ada validasi dev tersisa setelah itu - keputusannya di test.")
    else:
        print("   Berhenti di sini: dev tetap tidak dipakai untuk latih. Tulis hasilnya di reports/.")
else:
    print("2. Varian 'all' selesai. NB 08: CHECKPOINTS = %s @ %s (sha16 %s), arm beam 5."
          % (HF_REPO, CK.get("hf_revision"), CK.get("sha16")))
    print("   REFS paritas dihitung di laptop dari findings/%s/submission_dev_S.csv." % RUN_ID)

## Setelah notebook ini

1. **Salin blok `CLAUDE-HANDOFF-JSON`** ke chat. Isinya hanya agregat: WER per convo, CI,
   pergeseran bobot, kurva. Unduh `findings/10-dev-stage-finetune-c5/` ke repo seperti
   biasa (atau file `-lite.zip`). File `submission_dev_*.csv` dianalisis lokal oleh skrip yang
   hanya mencetak agregat.
2. **Check yang paling penting:** *"kontrol D = NB 08"*. Kalau PASS, lingkungan decode di sini
   sama dengan submission D, dan S − D bisa dibaca langsung. Kalau FAIL, perbandingan
   berpasangan tetap sah (keduanya di-decode di sesi ini), tapi angka S tidak bisa ditaruh di
   samping angka NB 08.
3. **Verdict:**
   - **JELAS LEBIH BAIK** → varian `all` (knob di sel 1), lalu NB 08 dan satu slot test
     melawan D 0,2901. Kurva greedy membantu memilih jumlah epoch untuk varian itu. Karena
     convo 2 jadi dipakai untuk memilih, catat ini di laporan.
   - **TIDAK JELAS / LEBIH BURUK** → berhenti. Tulis `reports/nb10-dev-stage-finetune-results.md`.
4. Kalau sesi mati **setelah** sel 9 mencetak revisi dan sha: set `RUN_MODE="eval"`,
   `EVAL_REVISION` dan `EVAL_SHA16` sesuai cetakan itu, lalu Run All.

**Deadline: 2 Okt 2026, 00:00 UTC.** Varian `all` + NB 08 + smoke + full butuh kira-kira
setengah hari kerja Kaggle. Jangan tunda keputusan setelah hasil ini keluar.